# Titanic: Machine Learning from Disaster

This notebook documents ten controlled experiments for the Kaggle Titanic competition.

The goal was to build a reproducible classification workflow, compare model families, test validation assumptions, and understand why apparently strong local improvements did or did not transfer to Kaggle.

**Champion model:** Random Forest with fold-validated family-outcome overrides

**Cross-validation accuracy:** 0.8390 

**Kaggle accuracy:** 0.79665

In [2]:
from pathlib import Path

import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

from xgboost import XGBClassifier


# ============================================================
# 1. LOCATE AND LOAD THE DATA
# ============================================================

# Works in either:
# 1. A Kaggle notebook with the Titanic dataset attached
# 2. A local project with files inside a folder named "data"

kaggle_data_path = Path("/kaggle/input/titanic")
local_data_path = Path("data")

if kaggle_data_path.exists():
    data_path = kaggle_data_path
    output_path = Path("/kaggle/working")
else:
    data_path = local_data_path
    output_path = Path(".")

train_path = data_path / "train.csv"
test_path = data_path / "test.csv"

if not train_path.exists() or not test_path.exists():
    raise FileNotFoundError(
        f"Could not find train.csv and test.csv in: {data_path}\n"
        "Place the files inside a folder named 'data', or attach the "
        "Titanic dataset to your Kaggle notebook."
    )

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)

print("Training data shape:", train.shape)
print("Test data shape:", test.shape)
print("\nTraining columns:")
print(train.columns.tolist())

Training data shape: (891, 12)
Test data shape: (418, 11)

Training columns:
['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked']


In [3]:
# ============================================================
# 2. QUICK DATA CHECKS
# ============================================================

print("\nTarget distribution:")
print(train["Survived"].value_counts(normalize=True).round(3))

print("\nMissing values in training data:")
print(
    train.isna()
    .sum()
    .sort_values(ascending=False)
    .loc[lambda x: x > 0]
)

print("\nMissing values in test data:")
print(
    test.isna()
    .sum()
    .sort_values(ascending=False)
    .loc[lambda x: x > 0]
)


# ============================================================
# 3. FEATURE ENGINEERING
# ============================================================

def add_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Create a small number of interpretable Titanic features.

    This function does not use Survived, so it can safely be applied
    to both the training data and the Kaggle test data.
    """
    df = df.copy()

    # Extract titles such as Mr, Mrs, Miss, and Master from Name.
    df["Title"] = (
        df["Name"]
        .str.extract(r",\s*([^.]*)\.", expand=False)
        .str.strip()
    )

    # Standardize equivalent titles.
    df["Title"] = df["Title"].replace(
        {
            "Mlle": "Miss",
            "Ms": "Miss",
            "Mme": "Mrs",
        }
    )

    # Combine uncommon titles to avoid creating many tiny categories.
    common_titles = ["Mr", "Mrs", "Miss", "Master"]
    df["Title"] = df["Title"].where(
        df["Title"].isin(common_titles),
        "Rare"
    )

    # Family members aboard, including the passenger.
    df["FamilySize"] = df["SibSp"] + df["Parch"] + 1

    # Whether the passenger was traveling alone.
    df["IsAlone"] = (df["FamilySize"] == 1).astype(int)

    # Use the first letter of Cabin as an approximate deck.
    df["Deck"] = df["Cabin"].fillna("Unknown").str[0]

    # Approximate fare per family member.
    df["FarePerPerson"] = df["Fare"] / df["FamilySize"]

    return df


train_features = add_features(train)
test_features = add_features(test)


# ============================================================
# 4. DEFINE THE MODELING DATA
# ============================================================

feature_columns = [
    "Pclass",
    "Sex",
    "Age",
    "SibSp",
    "Parch",
    "Fare",
    "Embarked",
    "Title",
    "FamilySize",
    "IsAlone",
    "Deck",
    "FarePerPerson",
]

numeric_features = [
    "Age",
    "SibSp",
    "Parch",
    "Fare",
    "FamilySize",
    "IsAlone",
    "FarePerPerson",
]

categorical_features = [
    "Pclass",
    "Sex",
    "Embarked",
    "Title",
    "Deck",
]

X = train_features[feature_columns]
y = train_features["Survived"]

X_test = test_features[feature_columns]


# ============================================================
# 5. PREPROCESSING
# ============================================================

numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
        ),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, categorical_features),
    ]
)



Target distribution:
Survived
0    0.616
1    0.384
Name: proportion, dtype: float64

Missing values in training data:
Cabin       687
Age         177
Embarked      2
dtype: int64

Missing values in test data:
Cabin    327
Age       86
Fare       1
dtype: int64


### EXP-001 — XGBoost Baseline

In [2]:

# ============================================================
# 6. XGBOOST MODEL
# ============================================================

model = XGBClassifier(
    n_estimators=300,
    learning_rate=0.03,
    max_depth=3,
    min_child_weight=2,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_alpha=0.10,
    reg_lambda=2.0,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1,
)

pipeline = Pipeline(
    steps=[
        ("preprocessing", preprocessor),
        ("model", model),
    ]
)


# ============================================================
# 7. CROSS-VALIDATION
# ============================================================

cross_validation = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

cv_scores = cross_val_score(
    pipeline,
    X,
    y,
    cv=cross_validation,
    scoring="accuracy",
)

print("\nCross-validation scores:")
print(cv_scores.round(4))

print(f"\nMean CV accuracy: {cv_scores.mean():.4f}")
print(f"CV standard deviation: {cv_scores.std():.4f}")


# ============================================================
# 8. TRAIN ON ALL LABELED DATA
# ============================================================

pipeline.fit(X, y)


# ============================================================
# 9. PREDICT THE KAGGLE TEST DATA
# ============================================================

test_predictions = pipeline.predict(X_test).astype(int)

print("\nPrediction distribution:")
print(pd.Series(test_predictions).value_counts().sort_index())


# ============================================================
# 10. CREATE THE SUBMISSION
# ============================================================

submission = pd.DataFrame(
    {
        "PassengerId": test["PassengerId"],
        "Survived": test_predictions,
    }
)

submission_path = output_path / "submission_xgb_baseline.csv"
submission.to_csv(submission_path, index=False)

print("\nSubmission preview:")
print(submission.head(10))

print("\nSubmission shape:", submission.shape)
print("Submission saved to:", submission_path)


# Final safety checks
assert submission.columns.tolist() == ["PassengerId", "Survived"]
assert len(submission) == len(test)
assert submission["PassengerId"].equals(test["PassengerId"])
assert submission["Survived"].isin([0, 1]).all()

print("\nAll submission checks passed.")

Training data shape: (891, 12)
Test data shape: (418, 11)

Training columns:
['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked']

Target distribution:
Survived
0    0.616
1    0.384
Name: proportion, dtype: float64

Missing values in training data:
Cabin       687
Age         177
Embarked      2
dtype: int64

Missing values in test data:
Cabin    327
Age       86
Fare       1
dtype: int64

Cross-validation scores:
[0.8659 0.8483 0.8034 0.8258 0.8427]

Mean CV accuracy: 0.8372
CV standard deviation: 0.0212

Prediction distribution:
0    262
1    156
Name: count, dtype: int64

Submission preview:
   PassengerId  Survived
0          892         0
1          893         0
2          894         0
3          895         0
4          896         1
5          897         0
6          898         1
7          899         0
8          900         1
9          901         0

Submission shape: (418, 2)
Submission saved to: C:\Users\

### EXP-002 — Logistic Regression Baseline


In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score

logistic_model = LogisticRegression(
    C=1.0,
    max_iter=2000,
    random_state=42,
)

logistic_pipeline = Pipeline(
    steps=[
        ("preprocessing", preprocessor),
        ("model", logistic_model),
    ]
)

logistic_cv_scores = cross_val_score(
    logistic_pipeline,
    X,
    y,
    cv=cross_validation,
    scoring="accuracy",
)

print("Logistic regression CV scores:")
print(logistic_cv_scores.round(4))

print(f"\nMean CV accuracy: {logistic_cv_scores.mean():.4f}")
print(f"CV standard deviation: {logistic_cv_scores.std():.4f}")

logistic_pipeline.fit(X, y)

logistic_predictions = logistic_pipeline.predict(X_test).astype(int)

logistic_submission = pd.DataFrame(
    {
        "PassengerId": test["PassengerId"],
        "Survived": logistic_predictions,
    }
)

logistic_submission_path = output_path / "submission_logistic.csv"
logistic_submission.to_csv(logistic_submission_path, index=False)

print("\nPrediction distribution:")
print(
    logistic_submission["Survived"]
    .value_counts()
    .sort_index()
)

print("\nSubmission preview:")
print(logistic_submission.head())

print("\nSaved to:")
print(logistic_submission_path)

assert logistic_submission.columns.tolist() == [
    "PassengerId",
    "Survived",
]
assert len(logistic_submission) == len(test)
assert logistic_submission["Survived"].isin([0, 1]).all()

Logistic regression CV scores:
[0.8436 0.8258 0.7978 0.8315 0.8427]

Mean CV accuracy: 0.8283
CV standard deviation: 0.0167

Prediction distribution:
Survived
0    250
1    168
Name: count, dtype: int64

Submission preview:
   PassengerId  Survived
0          892         0
1          893         1
2          894         0
3          895         0
4          896         1

Saved to:
C:\Users\Owner\Documents\Github\machine-learning-lab\00-Kaggle\01-Titanic - Machine Learning from Disaster\submission_logistic.csv


### EXP-003 — Random Forest Champion


In [9]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline
import pandas as pd


# ============================================================
# EXP-003: RANDOM FOREST
# ============================================================

random_forest_model = RandomForestClassifier(
    n_estimators=500,
    max_depth=5,
    min_samples_split=8,
    min_samples_leaf=4,
    max_features="sqrt",
    random_state=42,
    n_jobs=-1,
)

random_forest_pipeline = Pipeline(
    steps=[
        ("preprocessing", preprocessor),
        ("model", random_forest_model),
    ]
)

rf_cv_scores = cross_val_score(
    random_forest_pipeline,
    X,
    y,
    cv=cross_validation,
    scoring="accuracy",
)

print("Random Forest CV scores:")
print(rf_cv_scores.round(4))

print(f"\nMean CV accuracy: {rf_cv_scores.mean():.4f}")
print(f"CV standard deviation: {rf_cv_scores.std():.4f}")


# Train using all labeled passengers
random_forest_pipeline.fit(X, y)


# Predict the Kaggle test set
rf_predictions = (
    random_forest_pipeline
    .predict(X_test)
    .astype(int)
)


# Create submission
rf_submission = pd.DataFrame(
    {
        "PassengerId": test["PassengerId"],
        "Survived": rf_predictions,
    }
)

rf_submission_path = (
    output_path / "submission_random_forest.csv"
)

rf_submission.to_csv(
    rf_submission_path,
    index=False,
)


# Output checks
print("\nPrediction counts:")
print(
    rf_submission["Survived"]
    .value_counts()
    .sort_index()
)

print("\nSubmission preview:")
display(rf_submission.head())

print("\nSaved to:")
print(rf_submission_path)


assert rf_submission.columns.tolist() == [
    "PassengerId",
    "Survived",
]

assert len(rf_submission) == len(test)

assert rf_submission["PassengerId"].equals(
    test["PassengerId"]
)

assert rf_submission["Survived"].isin([0, 1]).all()

Random Forest CV scores:
[0.838  0.8202 0.8202 0.8315 0.8483]

Mean CV accuracy: 0.8316
CV standard deviation: 0.0108

Prediction counts:
Survived
0    259
1    159
Name: count, dtype: int64

Submission preview:


,PassengerId,Survived
0,892,0
1,893,0
2,894,0
3,895,0
4,896,1



Saved to:
C:\Users\Owner\Documents\Github\machine-learning-lab\00-Kaggle\01-Titanic - Machine Learning from Disaster\submission_random_forest.csv


In [10]:
model_comparison = pd.DataFrame(
    {
        "PassengerId": test["PassengerId"],
        "logistic": logistic_predictions,
        "random_forest": rf_predictions,
    }
)

model_comparison["disagree"] = (
    model_comparison["logistic"]
    != model_comparison["random_forest"]
)

print(
    "Differing test predictions:",
    model_comparison["disagree"].sum(),
)

display(
    model_comparison[
        model_comparison["disagree"]
    ].head(20)
)

Differing test predictions: 15


,PassengerId,logistic,random_forest,disagree
1,893,1,0,True
18,910,1,0,True
32,924,0,1,True
41,933,1,0,True
73,965,1,0,True
75,967,1,0,True
127,1019,0,1,True
146,1038,1,0,True
181,1073,1,0,True
242,1134,1,0,True


In [13]:
test_disagreements = test_features.loc[
    rf_predictions != logistic_predictions,
    [
        "PassengerId",
        "Name",
        "Sex",
        "Age",
        "Pclass",
        "Fare",
        "SibSp",
        "Parch",
        "Embarked",
        "Title",
        "FamilySize",
        "IsAlone",
        "Deck",
    ],
].copy()

test_disagreements["logistic_prediction"] = logistic_predictions[
    rf_predictions != logistic_predictions
]

test_disagreements["random_forest_prediction"] = rf_predictions[
    rf_predictions != logistic_predictions
]

test_disagreements["logistic_probability"] = (
    logistic_pipeline.predict_proba(X_test)[:, 1][
        rf_predictions != logistic_predictions
    ]
)

test_disagreements["random_forest_probability"] = (
    random_forest_pipeline.predict_proba(X_test)[:, 1][
        rf_predictions != logistic_predictions
    ]
)

display(
    test_disagreements.sort_values(
        "random_forest_probability",
        ascending=False,
    )
)

,PassengerId,Name,Sex,Age,Pclass,Fare,SibSp,Parch,Embarked,Title,FamilySize,IsAlone,Deck,logistic_prediction,random_forest_prediction,logistic_probability,random_forest_probability
127,1019,"McCoy, Miss. Alicia",female,NaN,3,23.2500,2,0,Q,Miss,3,0,U,0,1,0.489544,0.651755
32,924,"Dean, Mrs. Bertram (Eva Georgetta Light)",female,33.0,3,20.5750,1,2,S,Mrs,4,0,U,0,1,0.474766,0.549814
293,1185,"Dodge, Dr. Washington",male,53.0,1,81.8583,1,1,S,Rare,3,0,A,0,1,0.294045,0.516752
1,893,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,3,7.0000,1,0,S,Mrs,2,0,U,1,0,0.560250,0.499876
18,910,"Ilmakangas, Miss. Ida Livija",female,27.0,3,7.9250,1,0,S,Miss,2,0,U,1,0,0.541840,0.487551
181,1073,"Compton, Mr. Alexander Taylor Jr",male,37.0,1,83.1583,1,1,C,Mr,3,0,E,1,0,0.558077,0.462081
339,1231,"Betros, Master. Seman",male,NaN,3,7.2292,0,0,C,Master,1,1,U,1,0,0.712237,0.423276
73,965,"Ovies y Rodriguez, Mr. Servando",male,28.5,1,27.7208,0,0,C,Mr,1,1,D,1,0,0.638612,0.416571
41,933,"Franklin, Mr. Thomas Parham",male,NaN,1,26.5500,0,0,S,Mr,1,1,D,1,0,0.540525,0.409378
242,1134,"Spedden, Mr. Frederic Oakley",male,45.0,1,134.5000,1,1,C,Mr,3,0,E,1,0,0.555280,0.407859


In [14]:
share_columns = [
    "PassengerId",
    "Sex",
    "Age",
    "Pclass",
    "Fare",
    "Title",
    "FamilySize",
    "IsAlone",
    "Deck",
    "logistic_prediction",
    "random_forest_prediction",
    "logistic_probability",
    "random_forest_probability",
]

print(
    test_disagreements[share_columns]
    .round(
        {
            "Age": 1,
            "Fare": 2,
            "logistic_probability": 3,
            "random_forest_probability": 3,
        }
    )
    .to_string(index=False)
)

 PassengerId    Sex  Age  Pclass   Fare  Title  FamilySize  IsAlone Deck  logistic_prediction  random_forest_prediction  logistic_probability  random_forest_probability
         893 female 47.0       3   7.00    Mrs           2        0    U                    1                         0                 0.560                      0.500
         910 female 27.0       3   7.92   Miss           2        0    U                    1                         0                 0.542                      0.488
         924 female 33.0       3  20.58    Mrs           4        0    U                    0                         1                 0.475                      0.550
         933   male  NaN       1  26.55     Mr           1        1    D                    1                         0                 0.541                      0.409
         965   male 28.5       1  27.72     Mr           1        1    D                    1                         0                 0.639              

### EXP-004 — Logistic Regression with Interactions


In [16]:
# ============================================================
# EXP-004: LOGISTIC REGRESSION WITH INTERACTION FEATURES
# ============================================================

import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline


# ------------------------------------------------------------
# 1. ADD TARGETED INTERACTION FEATURES
# ------------------------------------------------------------

def add_interaction_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Passenger class may affect men and women differently.
    df["Sex_Pclass"] = (
        df["Sex"].astype(str)
        + "_class_"
        + df["Pclass"].astype(str)
    )

    # Titles such as Mr, Mrs, Miss, and Master interact strongly with sex.
    df["Sex_Title"] = (
        df["Sex"].astype(str)
        + "_"
        + df["Title"].astype(str)
    )

    # Class may affect different titles differently.
    df["Title_Pclass"] = (
        df["Title"].astype(str)
        + "_class_"
        + df["Pclass"].astype(str)
    )

    return df


train_interactions = add_interaction_features(train_features)
test_interactions = add_interaction_features(test_features)


# ------------------------------------------------------------
# 2. UPDATE FEATURE LISTS
# ------------------------------------------------------------

interaction_features = [
    "Sex_Pclass",
    "Sex_Title",
    "Title_Pclass",
]

interaction_feature_columns = (
    feature_columns
    + interaction_features
)

interaction_categorical_features = (
    categorical_features
    + interaction_features
)

X_interactions = train_interactions[
    interaction_feature_columns
]

X_test_interactions = test_interactions[
    interaction_feature_columns
]


# ------------------------------------------------------------
# 3. CREATE A NEW PREPROCESSOR
# ------------------------------------------------------------

interaction_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_pipeline,
            numeric_features,
        ),
        (
            "categorical",
            categorical_pipeline,
            interaction_categorical_features,
        ),
    ]
)


# ------------------------------------------------------------
# 4. LOGISTIC REGRESSION
# ------------------------------------------------------------

logistic_interaction_model = LogisticRegression(
    C=1.0,
    max_iter=2000,
    random_state=42,
)

logistic_interaction_pipeline = Pipeline(
    steps=[
        (
            "preprocessing",
            interaction_preprocessor,
        ),
        (
            "model",
            logistic_interaction_model,
        ),
    ]
)


# ------------------------------------------------------------
# 5. CROSS-VALIDATION
# ------------------------------------------------------------

logistic_interaction_cv_scores = cross_val_score(
    logistic_interaction_pipeline,
    X_interactions,
    y,
    cv=cross_validation,
    scoring="accuracy",
)

print("Interaction Logistic CV scores:")
print(
    logistic_interaction_cv_scores.round(4)
)

print(
    f"\nMean CV accuracy: "
    f"{logistic_interaction_cv_scores.mean():.4f}"
)

print(
    f"CV standard deviation: "
    f"{logistic_interaction_cv_scores.std():.4f}"
)


# ------------------------------------------------------------
# 6. TRAIN AND PREDICT
# ------------------------------------------------------------

logistic_interaction_pipeline.fit(
    X_interactions,
    y,
)

logistic_interaction_predictions = (
    logistic_interaction_pipeline
    .predict(X_test_interactions)
    .astype(int)
)


# ------------------------------------------------------------
# 7. CREATE SUBMISSION
# ------------------------------------------------------------

interaction_submission = pd.DataFrame(
    {
        "PassengerId": test["PassengerId"],
        "Survived": logistic_interaction_predictions,
    }
)

interaction_submission_path = (
    output_path
    / "submission_logistic_interactions.csv"
)

interaction_submission.to_csv(
    interaction_submission_path,
    index=False,
)

print("\nPrediction counts:")
print(
    interaction_submission["Survived"]
    .value_counts()
    .sort_index()
)

print("\nSubmission preview:")
display(interaction_submission.head())

print("\nSaved to:")
print(interaction_submission_path)


# ------------------------------------------------------------
# 8. SAFETY CHECKS
# ------------------------------------------------------------

assert interaction_submission.columns.tolist() == [
    "PassengerId",
    "Survived",
]

assert len(interaction_submission) == len(test)

assert interaction_submission[
    "PassengerId"
].equals(test["PassengerId"])

assert interaction_submission[
    "Survived"
].isin([0, 1]).all()

Interaction Logistic CV scores:
[0.8268 0.8539 0.7978 0.8427 0.8483]

Mean CV accuracy: 0.8339
CV standard deviation: 0.0202

Prediction counts:
Survived
0    261
1    157
Name: count, dtype: int64

Submission preview:


,PassengerId,Survived
0,892,0
1,893,0
2,894,0
3,895,0
4,896,1



Saved to:
C:\Users\Owner\Documents\Github\machine-learning-lab\00-Kaggle\01-Titanic - Machine Learning from Disaster\submission_logistic_interactions.csv


In [17]:
comparison_exp_004 = pd.DataFrame(
    {
        "PassengerId": test["PassengerId"],
        "logistic_baseline": logistic_predictions,
        "logistic_interactions": logistic_interaction_predictions,
        "random_forest": rf_predictions,
    }
)

print(
    "Interaction logistic vs baseline logistic:",
    (
        comparison_exp_004["logistic_interactions"]
        != comparison_exp_004["logistic_baseline"]
    ).sum(),
)

print(
    "Interaction logistic vs random forest:",
    (
        comparison_exp_004["logistic_interactions"]
        != comparison_exp_004["random_forest"]
    ).sum(),
)

Interaction logistic vs baseline logistic: 11
Interaction logistic vs random forest: 16


### EXP-005 — Group-Based Age Imputation


In [20]:
# ============================================================
# EXP-005: RANDOM FOREST WITH GROUP-BASED AGE IMPUTATION
# ============================================================

import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline


# ------------------------------------------------------------
# 1. CUSTOM AGE IMPUTER
# ------------------------------------------------------------

class GroupedAgeImputer(BaseEstimator, TransformerMixin):
    """
    Impute missing Age using medians learned from training data.

    Fallback order:
    1. Title + Pclass median
    2. Title median
    3. Pclass median
    4. Overall median
    """

    def fit(self, X, y=None):
        X = X.copy()

        self.title_class_medians_ = (
            X.groupby(["Title", "Pclass"])["Age"]
            .median()
            .to_dict()
        )

        self.title_medians_ = (
            X.groupby("Title")["Age"]
            .median()
            .to_dict()
        )

        self.class_medians_ = (
            X.groupby("Pclass")["Age"]
            .median()
            .to_dict()
        )

        self.overall_median_ = X["Age"].median()

        return self

    def transform(self, X):
        X = X.copy()

        missing_age_rows = X["Age"].isna()

        for row_index in X.index[missing_age_rows]:
            title = X.at[row_index, "Title"]
            passenger_class = X.at[row_index, "Pclass"]

            estimated_age = self.title_class_medians_.get(
                (title, passenger_class),
                np.nan,
            )

            if pd.isna(estimated_age):
                estimated_age = self.title_medians_.get(
                    title,
                    np.nan,
                )

            if pd.isna(estimated_age):
                estimated_age = self.class_medians_.get(
                    passenger_class,
                    np.nan,
                )

            if pd.isna(estimated_age):
                estimated_age = self.overall_median_

            X.at[row_index, "Age"] = estimated_age

        return X


# ------------------------------------------------------------
# 2. SAME RANDOM FOREST SETTINGS AS EXP-003
# ------------------------------------------------------------

rf_grouped_age_model = RandomForestClassifier(
    n_estimators=500,
    max_depth=5,
    min_samples_split=8,
    min_samples_leaf=4,
    max_features="sqrt",
    random_state=42,
    n_jobs=-1,
)


# ------------------------------------------------------------
# 3. PIPELINE
# ------------------------------------------------------------

rf_grouped_age_pipeline = Pipeline(
    steps=[
        (
            "grouped_age_imputer",
            GroupedAgeImputer(),
        ),
        (
            "preprocessing",
            preprocessor,
        ),
        (
            "model",
            rf_grouped_age_model,
        ),
    ]
)


# ------------------------------------------------------------
# 4. CROSS-VALIDATION
# ------------------------------------------------------------

rf_grouped_age_cv_scores = cross_val_score(
    rf_grouped_age_pipeline,
    X,
    y,
    cv=cross_validation,
    scoring="accuracy",
)

print("Grouped-Age Random Forest CV scores:")
print(rf_grouped_age_cv_scores.round(4))

print(
    f"\nMean CV accuracy: "
    f"{rf_grouped_age_cv_scores.mean():.4f}"
)

print(
    f"CV standard deviation: "
    f"{rf_grouped_age_cv_scores.std():.4f}"
)


# ------------------------------------------------------------
# 5. TRAIN ON ALL LABELED DATA
# ------------------------------------------------------------

rf_grouped_age_pipeline.fit(X, y)


# ------------------------------------------------------------
# 6. PREDICT TEST DATA
# ------------------------------------------------------------

rf_grouped_age_predictions = (
    rf_grouped_age_pipeline
    .predict(X_test)
    .astype(int)
)


# ------------------------------------------------------------
# 7. CREATE SUBMISSION
# ------------------------------------------------------------

rf_grouped_age_submission = pd.DataFrame(
    {
        "PassengerId": test["PassengerId"],
        "Survived": rf_grouped_age_predictions,
    }
)

rf_grouped_age_submission_path = (
    output_path
    / "submission_rf_grouped_age.csv"
)

rf_grouped_age_submission.to_csv(
    rf_grouped_age_submission_path,
    index=False,
)

print("\nPrediction counts:")
print(
    rf_grouped_age_submission["Survived"]
    .value_counts()
    .sort_index()
)

print("\nSubmission preview:")
display(rf_grouped_age_submission.head())

print("\nSaved to:")
print(rf_grouped_age_submission_path)


# ------------------------------------------------------------
# 8. SAFETY CHECKS
# ------------------------------------------------------------

assert rf_grouped_age_submission.columns.tolist() == [
    "PassengerId",
    "Survived",
]

assert len(rf_grouped_age_submission) == len(test)

assert rf_grouped_age_submission[
    "PassengerId"
].equals(test["PassengerId"])

assert rf_grouped_age_submission[
    "Survived"
].isin([0, 1]).all()

Grouped-Age Random Forest CV scores:
[0.838  0.8146 0.8258 0.8315 0.8371]

Mean CV accuracy: 0.8294
CV standard deviation: 0.0086

Prediction counts:
Survived
0    255
1    163
Name: count, dtype: int64

Submission preview:


,PassengerId,Survived
0,892,0
1,893,1
2,894,0
3,895,0
4,896,1



Saved to:
C:\Users\Owner\Documents\Github\machine-learning-lab\00-Kaggle\01-Titanic - Machine Learning from Disaster\submission_rf_grouped_age.csv


In [21]:
exp_005_comparison = pd.DataFrame(
    {
        "PassengerId": test["PassengerId"],
        "rf_baseline": rf_predictions,
        "rf_grouped_age": rf_grouped_age_predictions,
    }
)

exp_005_comparison["disagree"] = (
    exp_005_comparison["rf_baseline"]
    != exp_005_comparison["rf_grouped_age"]
)

print(
    "Differing predictions versus EXP-003:",
    exp_005_comparison["disagree"].sum(),
)

display(
    exp_005_comparison[
        exp_005_comparison["disagree"]
    ]
)

Differing predictions versus EXP-003: 4


,PassengerId,rf_baseline,rf_grouped_age,disagree
1,893,0,1,True
244,1136,0,1,True
339,1231,0,1,True
344,1236,0,1,True


##### EXP-006

The ordinary StratifiedKFold validation may be optimistic because related passengers can appear in both the training and validation portions of a fold.

For example, the model could train on one member of the Andersson family and validate on another member with similar:

Surname
Fare
Ticket characteristics
Family size
Passenger class

Group-aware validation forces likely family members to remain together.

What remains constant
The EXP-003 Random Forest
All model parameters
All features
All preprocessing
Five validation folds
Accuracy as the evaluation metric
The complete Titanic training dataset
What changes

Only the validation split:

Current method: StratifiedKFold
New method: StratifiedGroupKFold
Group definition: surname plus family size
Passengers travelling alone receive unique groups
What supports or rejects the hypothesis

Supports the hypothesis:

Group-aware CV is around 0.01 or more below ordinary CV
Most grouped folds perform worse
Out-of-fold predictions change meaningfully

That would suggest family leakage was inflating our previous scores.

Rejects the hypothesis:

Group-aware CV remains very close to ordinary CV, roughly within 0.005
Fold stability remains similar
Few out-of-fold predictions change

That would suggest family overlap is not the main reason for the CV–Kaggle gap.

A result between those ranges would be inconclusive rather than a clean win or loss.

### EXP-006 — Group-Aware Validation


In [23]:
# ============================================================
# EXP-006: GROUP-AWARE VALIDATION
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.metrics import accuracy_score
from sklearn.model_selection import (
    StratifiedGroupKFold,
    cross_val_predict,
    cross_val_score,
)


# ------------------------------------------------------------
# 1. CREATE FAMILY GROUPS
# ------------------------------------------------------------

validation_data = train_features.copy()

validation_data["Surname"] = (
    validation_data["Name"]
    .str.extract(r"^([^,]+),", expand=False)
    .str.strip()
    .str.lower()
)

# Likely family members share surname and family size.
#
# Passengers travelling alone receive unique groups so that
# unrelated solo passengers with the same surname are not grouped.

validation_data["FamilyGroup"] = np.where(
    validation_data["FamilySize"] > 1,
    (
        validation_data["Surname"]
        + "_family_"
        + validation_data["FamilySize"].astype(int).astype(str)
    ),
    (
        "solo_passenger_"
        + validation_data["PassengerId"].astype(str)
    ),
)

family_groups = validation_data["FamilyGroup"]

print("Number of passengers:", len(validation_data))
print("Number of family groups:", family_groups.nunique())

print(
    "Passengers in multi-person family groups:",
    (validation_data["FamilySize"] > 1).sum(),
)

print(
    "Passengers travelling alone:",
    (validation_data["FamilySize"] == 1).sum(),
)


# ------------------------------------------------------------
# 2. DEFINE GROUP-AWARE VALIDATION
# ------------------------------------------------------------

group_cross_validation = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)


# ------------------------------------------------------------
# 3. CONFIRM THAT GROUPS DO NOT CROSS FOLDS
# ------------------------------------------------------------

fold_summaries = []

for fold_number, (train_indices, valid_indices) in enumerate(
    group_cross_validation.split(
        X,
        y,
        groups=family_groups,
    ),
    start=1,
):
    train_group_values = set(
        family_groups.iloc[train_indices]
    )

    valid_group_values = set(
        family_groups.iloc[valid_indices]
    )

    overlapping_groups = (
        train_group_values
        & valid_group_values
    )

    fold_summaries.append(
        {
            "fold": fold_number,
            "training_rows": len(train_indices),
            "validation_rows": len(valid_indices),
            "validation_survival_rate": (
                y.iloc[valid_indices].mean()
            ),
            "validation_groups": len(
                valid_group_values
            ),
            "group_overlap": len(
                overlapping_groups
            ),
        }
    )

fold_summary = pd.DataFrame(fold_summaries)

print("\nGrouped-fold summary:")
display(fold_summary.round(4))

assert fold_summary["group_overlap"].eq(0).all()


# ------------------------------------------------------------
# 4. ORDINARY STRATIFIED VALIDATION
# ------------------------------------------------------------

ordinary_cv_scores = cross_val_score(
    random_forest_pipeline,
    X,
    y,
    cv=cross_validation,
    scoring="accuracy",
)

print("\nOrdinary StratifiedKFold scores:")
print(ordinary_cv_scores.round(4))

print(
    f"Ordinary mean accuracy: "
    f"{ordinary_cv_scores.mean():.4f}"
)

print(
    f"Ordinary standard deviation: "
    f"{ordinary_cv_scores.std():.4f}"
)


# ------------------------------------------------------------
# 5. GROUP-AWARE VALIDATION
# ------------------------------------------------------------

grouped_cv_scores = cross_val_score(
    random_forest_pipeline,
    X,
    y,
    cv=group_cross_validation,
    groups=family_groups,
    scoring="accuracy",
)

print("\nStratifiedGroupKFold scores:")
print(grouped_cv_scores.round(4))

print(
    f"Grouped mean accuracy: "
    f"{grouped_cv_scores.mean():.4f}"
)

print(
    f"Grouped standard deviation: "
    f"{grouped_cv_scores.std():.4f}"
)


# ------------------------------------------------------------
# 6. OUT-OF-FOLD PREDICTIONS
# ------------------------------------------------------------

ordinary_oof_predictions = cross_val_predict(
    random_forest_pipeline,
    X,
    y,
    cv=cross_validation,
    method="predict",
)

grouped_oof_predictions = cross_val_predict(
    random_forest_pipeline,
    X,
    y,
    cv=group_cross_validation,
    groups=family_groups,
    method="predict",
)

ordinary_oof_accuracy = accuracy_score(
    y,
    ordinary_oof_predictions,
)

grouped_oof_accuracy = accuracy_score(
    y,
    grouped_oof_predictions,
)

oof_disagreements = (
    ordinary_oof_predictions
    != grouped_oof_predictions
).sum()

print(
    f"\nOrdinary OOF accuracy: "
    f"{ordinary_oof_accuracy:.4f}"
)

print(
    f"Grouped OOF accuracy: "
    f"{grouped_oof_accuracy:.4f}"
)

print(
    "Differing out-of-fold predictions:",
    oof_disagreements,
)


# ------------------------------------------------------------
# 7. SUMMARIZE THE VALIDATION COMPARISON
# ------------------------------------------------------------

validation_comparison = pd.DataFrame(
    [
        {
            "validation_method": (
                "5-fold StratifiedKFold"
            ),
            "fold_scores": ", ".join(
                f"{score:.4f}"
                for score in ordinary_cv_scores
            ),
            "mean_accuracy": (
                ordinary_cv_scores.mean()
            ),
            "cv_std": ordinary_cv_scores.std(),
            "oof_accuracy": ordinary_oof_accuracy,
        },
        {
            "validation_method": (
                "5-fold StratifiedGroupKFold"
            ),
            "fold_scores": ", ".join(
                f"{score:.4f}"
                for score in grouped_cv_scores
            ),
            "mean_accuracy": (
                grouped_cv_scores.mean()
            ),
            "cv_std": grouped_cv_scores.std(),
            "oof_accuracy": grouped_oof_accuracy,
        },
    ]
)

ordinary_mean = ordinary_cv_scores.mean()
grouped_mean = grouped_cv_scores.mean()

validation_difference = (
    grouped_mean - ordinary_mean
)

print(
    "\nGrouped minus ordinary mean accuracy:",
    f"{validation_difference:+.4f}",
)

display(
    validation_comparison.style.format(
        {
            "mean_accuracy": "{:.4f}",
            "cv_std": "{:.4f}",
            "oof_accuracy": "{:.4f}",
        }
    )
)


# ------------------------------------------------------------
# 8. SAVE THE VALIDATION RESULTS
# ------------------------------------------------------------

validation_results_path = (
    output_path / "validation_exp006.csv"
)

validation_comparison.to_csv(
    validation_results_path,
    index=False,
)

print("\nSaved validation comparison to:")
print(validation_results_path)

Number of passengers: 891
Number of family groups: 739
Passengers in multi-person family groups: 354
Passengers travelling alone: 537

Grouped-fold summary:


,fold,training_rows,validation_rows,validation_survival_rate,validation_groups,group_overlap
0,1,716,175,0.4114,144,0
1,2,714,177,0.3390,146,0
2,3,717,174,0.3908,149,0
3,4,703,188,0.4096,150,0
4,5,714,177,0.3672,150,0



Ordinary StratifiedKFold scores:
[0.838  0.8202 0.8202 0.8315 0.8483]
Ordinary mean accuracy: 0.8316
Ordinary standard deviation: 0.0108

StratifiedGroupKFold scores:
[0.7486 0.8588 0.8276 0.8245 0.8757]
Grouped mean accuracy: 0.8270
Grouped standard deviation: 0.0437

Ordinary OOF accuracy: 0.8316
Grouped OOF accuracy: 0.8272
Differing out-of-fold predictions: 12

Grouped minus ordinary mean accuracy: -0.0046


,validation_method,fold_scores,mean_accuracy,cv_std,oof_accuracy
0,5-fold StratifiedKFold,"0.8380, 0.8202, 0.8202, 0.8315, 0.8483",0.8316,0.0108,0.8316
1,5-fold StratifiedGroupKFold,"0.7486, 0.8588, 0.8276, 0.8245, 0.8757",0.8270,0.0437,0.8272



Saved validation comparison to:
C:\Users\Owner\Documents\Github\machine-learning-lab\00-Kaggle\01-Titanic - Machine Learning from Disaster\validation_exp006.csv


##### EXP-007

### EXP-007 — Random Forest Hyperparameter Tuning


In [25]:
# ============================================================
# EXP-007: CONSERVATIVE RANDOM FOREST TUNING
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.model_selection import (
    GridSearchCV,
    RepeatedStratifiedKFold,
    StratifiedGroupKFold,
    cross_val_score,
)


# ------------------------------------------------------------
# 1. DEFINE THE EXP-003 CHAMPION SETTINGS
# ------------------------------------------------------------

baseline_params = {
    "model__max_depth": 5,
    "model__min_samples_leaf": 4,
    "model__max_features": "sqrt",
    "model__min_samples_split": 8,
}


# ------------------------------------------------------------
# 2. DEFINE A SMALL PREDETERMINED SEARCH GRID
# ------------------------------------------------------------

parameter_grid = {
    "model__max_depth": [4, 5, 6],
    "model__min_samples_leaf": [3, 4, 6],
    "model__max_features": ["sqrt", 0.75],

    # Held constant from EXP-003.
    "model__min_samples_split": [8],
}


# ------------------------------------------------------------
# 3. INITIAL FIVE-FOLD SCREEN
# ------------------------------------------------------------

grid_search = GridSearchCV(
    estimator=clone(random_forest_pipeline),
    param_grid=parameter_grid,
    scoring="accuracy",
    cv=cross_validation,
    n_jobs=-1,
    refit=False,
    return_train_score=False,
)

grid_search.fit(X, y)

grid_results = pd.DataFrame(
    grid_search.cv_results_
).sort_values(
    ["mean_test_score", "std_test_score"],
    ascending=[False, True],
).reset_index(drop=True)

grid_results["rank"] = np.arange(
    1,
    len(grid_results) + 1,
)

grid_display_columns = [
    "rank",
    "mean_test_score",
    "std_test_score",
    "param_model__max_depth",
    "param_model__min_samples_leaf",
    "param_model__max_features",
    "param_model__min_samples_split",
]

print("Top grid-search results:")

display(
    grid_results[grid_display_columns]
    .head(10)
    .style.format(
        {
            "mean_test_score": "{:.4f}",
            "std_test_score": "{:.4f}",
        }
    )
)


# ------------------------------------------------------------
# 4. SAVE COMPLETE GRID RESULTS
# ------------------------------------------------------------

results_directory = output_path / "results"
results_directory.mkdir(
    parents=True,
    exist_ok=True,
)

grid_results_path = (
    results_directory
    / "exp007_rf_grid_results.csv"
)

grid_results.to_csv(
    grid_results_path,
    index=False,
)

print("\nFull grid results saved to:")
print(grid_results_path)


# ------------------------------------------------------------
# 5. SELECT TOP THREE CONFIGURATIONS
# ------------------------------------------------------------

top_three_params = (
    grid_results
    .head(3)["params"]
    .tolist()
)


def parameter_signature(params):
    """
    Convert a parameter dictionary into a stable signature
    so duplicate configurations can be removed.
    """
    return tuple(
        sorted(
            (key, str(value))
            for key, value in params.items()
        )
    )


candidate_param_sets = [
    baseline_params,
    *top_three_params,
]

unique_candidates = []
seen_signatures = set()

for params in candidate_param_sets:
    signature = parameter_signature(params)

    if signature not in seen_signatures:
        seen_signatures.add(signature)
        unique_candidates.append(params)


# ------------------------------------------------------------
# 6. REPEATED CROSS-VALIDATION
# ------------------------------------------------------------

repeated_cv = RepeatedStratifiedKFold(
    n_splits=5,
    n_repeats=5,
    random_state=42,
)

finalist_rows = []

for candidate_number, params in enumerate(
    unique_candidates,
    start=1,
):
    candidate_pipeline = (
        clone(random_forest_pipeline)
        .set_params(**params)
    )

    repeated_scores = cross_val_score(
        candidate_pipeline,
        X,
        y,
        cv=repeated_cv,
        scoring="accuracy",
        n_jobs=-1,
    )

    is_baseline = (
        parameter_signature(params)
        == parameter_signature(baseline_params)
    )

    finalist_rows.append(
        {
            "candidate": (
                "EXP-003 baseline"
                if is_baseline
                else f"Tuned candidate {candidate_number}"
            ),
            "params": params,
            "repeated_cv_mean": repeated_scores.mean(),
            "repeated_cv_std": repeated_scores.std(),
            "repeated_cv_min": repeated_scores.min(),
            "repeated_cv_max": repeated_scores.max(),
            "number_of_scores": len(repeated_scores),
        }
    )

finalist_results = pd.DataFrame(
    finalist_rows
).sort_values(
    [
        "repeated_cv_mean",
        "repeated_cv_std",
    ],
    ascending=[
        False,
        True,
    ],
).reset_index(drop=True)

print("\nRepeated cross-validation finalists:")

display(
    finalist_results[
        [
            "candidate",
            "repeated_cv_mean",
            "repeated_cv_std",
            "repeated_cv_min",
            "repeated_cv_max",
            "params",
        ]
    ].style.format(
        {
            "repeated_cv_mean": "{:.4f}",
            "repeated_cv_std": "{:.4f}",
            "repeated_cv_min": "{:.4f}",
            "repeated_cv_max": "{:.4f}",
        }
    )
)


# ------------------------------------------------------------
# 7. IDENTIFY BASELINE AND BEST CANDIDATE
# ------------------------------------------------------------

baseline_signature = parameter_signature(
    baseline_params
)

baseline_result = next(
    row
    for row in finalist_rows
    if parameter_signature(row["params"])
    == baseline_signature
)

best_result = finalist_results.iloc[0]

best_params = best_result["params"]

best_repeated_mean = (
    best_result["repeated_cv_mean"]
)

best_repeated_std = (
    best_result["repeated_cv_std"]
)

baseline_repeated_mean = (
    baseline_result["repeated_cv_mean"]
)

baseline_repeated_std = (
    baseline_result["repeated_cv_std"]
)

repeated_cv_gain = (
    best_repeated_mean
    - baseline_repeated_mean
)

best_is_baseline = (
    parameter_signature(best_params)
    == baseline_signature
)

print("\nBest repeated-CV parameters:")
print(best_params)

print(
    "\nRepeated-CV gain versus EXP-003:",
    f"{repeated_cv_gain:+.4f}",
)


# ------------------------------------------------------------
# 8. RECREATE FAMILY GROUPS FOR ROBUSTNESS CHECK
# ------------------------------------------------------------

validation_data_exp007 = train_features.copy()

validation_data_exp007["Surname"] = (
    validation_data_exp007["Name"]
    .str.extract(
        r"^([^,]+),",
        expand=False,
    )
    .str.strip()
    .str.lower()
)

validation_data_exp007["FamilyGroup"] = np.where(
    validation_data_exp007["FamilySize"] > 1,
    (
        validation_data_exp007["Surname"]
        + "_family_"
        + validation_data_exp007[
            "FamilySize"
        ].astype(int).astype(str)
    ),
    (
        "solo_passenger_"
        + validation_data_exp007[
            "PassengerId"
        ].astype(str)
    ),
)

family_groups_exp007 = (
    validation_data_exp007["FamilyGroup"]
)

group_cv_exp007 = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)


# ------------------------------------------------------------
# 9. GROUP-AWARE ROBUSTNESS CHECK
# ------------------------------------------------------------

baseline_pipeline_exp007 = (
    clone(random_forest_pipeline)
    .set_params(**baseline_params)
)

best_pipeline_exp007 = (
    clone(random_forest_pipeline)
    .set_params(**best_params)
)

baseline_group_scores = cross_val_score(
    baseline_pipeline_exp007,
    X,
    y,
    cv=group_cv_exp007,
    groups=family_groups_exp007,
    scoring="accuracy",
    n_jobs=-1,
)

best_group_scores = cross_val_score(
    best_pipeline_exp007,
    X,
    y,
    cv=group_cv_exp007,
    groups=family_groups_exp007,
    scoring="accuracy",
    n_jobs=-1,
)

grouped_cv_gain = (
    best_group_scores.mean()
    - baseline_group_scores.mean()
)

print("\nEXP-003 grouped scores:")
print(baseline_group_scores.round(4))

print(
    "EXP-003 grouped mean:",
    f"{baseline_group_scores.mean():.4f}",
)

print("\nBest candidate grouped scores:")
print(best_group_scores.round(4))

print(
    "Best candidate grouped mean:",
    f"{best_group_scores.mean():.4f}",
)

print(
    "\nGrouped-CV gain versus EXP-003:",
    f"{grouped_cv_gain:+.4f}",
)


# ------------------------------------------------------------
# 10. DETERMINE WHETHER SUBMISSION IS JUSTIFIED
# ------------------------------------------------------------

meaningful_repeated_gain = (
    repeated_cv_gain >= 0.005
)

acceptable_stability = (
    best_repeated_std
    <= baseline_repeated_std + 0.005
)

acceptable_grouped_result = (
    grouped_cv_gain >= -0.005
)

candidate_differs_from_baseline = (
    not best_is_baseline
)

submission_justified = all(
    [
        meaningful_repeated_gain,
        acceptable_stability,
        acceptable_grouped_result,
        candidate_differs_from_baseline,
    ]
)

decision_summary = pd.DataFrame(
    [
        {
            "check": (
                "Repeated CV gain >= 0.005"
            ),
            "result": meaningful_repeated_gain,
            "observed": repeated_cv_gain,
        },
        {
            "check": (
                "CV stability remains acceptable"
            ),
            "result": acceptable_stability,
            "observed": (
                best_repeated_std
                - baseline_repeated_std
            ),
        },
        {
            "check": (
                "Grouped CV gain >= -0.005"
            ),
            "result": acceptable_grouped_result,
            "observed": grouped_cv_gain,
        },
        {
            "check": (
                "Parameters differ from baseline"
            ),
            "result": (
                candidate_differs_from_baseline
            ),
            "observed": np.nan,
        },
    ]
)

print("\nSubmission decision checks:")
display(decision_summary)

print(
    "\nSubmission justified:",
    submission_justified,
)


# ------------------------------------------------------------
# 11. SAVE FINALIST RESULTS
# ------------------------------------------------------------

finalist_results_path = (
    results_directory
    / "exp007_rf_finalists.csv"
)

finalist_results.to_csv(
    finalist_results_path,
    index=False,
)

print("\nFinalist results saved to:")
print(finalist_results_path)


# ------------------------------------------------------------
# 12. CREATE A SUBMISSION ONLY IF JUSTIFIED
# ------------------------------------------------------------

if submission_justified:
    best_pipeline_exp007.fit(X, y)

    exp007_predictions = (
        best_pipeline_exp007
        .predict(X_test)
        .astype(int)
    )

    exp007_submission = pd.DataFrame(
        {
            "PassengerId": test["PassengerId"],
            "Survived": exp007_predictions,
        }
    )

    exp007_submission_path = (
        output_path
        / "submission_rf_tuned_exp007.csv"
    )

    exp007_submission.to_csv(
        exp007_submission_path,
        index=False,
    )

    print("\nSubmission created:")
    print(exp007_submission_path)

    print("\nPrediction counts:")
    print(
        exp007_submission["Survived"]
        .value_counts()
        .sort_index()
    )

    if "rf_predictions" in globals():
        exp007_disagreements = (
            exp007_predictions
            != rf_predictions
        ).sum()

        print(
            "\nDiffering predictions versus EXP-003:",
            exp007_disagreements,
        )

else:
    print(
        "\nNo Kaggle submission recommended. "
        "EXP-003 remains the champion."
    )

Top grid-search results:


,rank,mean_test_score,std_test_score,param_model__max_depth,param_model__min_samples_leaf,param_model__max_features,param_model__min_samples_split
0,1,0.8440,0.0155,5,6,0.750000,8
1,2,0.8428,0.0157,5,4,0.750000,8
2,3,0.8428,0.0121,6,6,0.750000,8
3,4,0.8417,0.0117,6,4,0.750000,8
4,5,0.8395,0.0178,5,3,0.750000,8
5,6,0.8395,0.0182,6,3,0.750000,8
6,7,0.8361,0.0116,6,3,sqrt,8
7,8,0.8339,0.0060,4,6,0.750000,8
8,9,0.8339,0.0079,4,3,0.750000,8
9,10,0.8339,0.0087,4,4,0.750000,8



Full grid results saved to:
C:\Users\Owner\Documents\Github\machine-learning-lab\00-Kaggle\01-Titanic - Machine Learning from Disaster\results\exp007_rf_grid_results.csv

Repeated cross-validation finalists:


,candidate,repeated_cv_mean,repeated_cv_std,repeated_cv_min,repeated_cv_max,params
0,Tuned candidate 2,0.8350,0.0214,0.7978,0.8764,"{'model__max_depth': 5, 'model__max_features': 0.75, 'model__min_samples_leaf': 6, 'model__min_samples_split': 8}"
1,Tuned candidate 4,0.8348,0.0193,0.7978,0.8708,"{'model__max_depth': 6, 'model__max_features': 0.75, 'model__min_samples_leaf': 6, 'model__min_samples_split': 8}"
2,Tuned candidate 3,0.8339,0.0233,0.7865,0.8764,"{'model__max_depth': 5, 'model__max_features': 0.75, 'model__min_samples_leaf': 4, 'model__min_samples_split': 8}"
3,EXP-003 baseline,0.8271,0.0236,0.7809,0.8764,"{'model__max_depth': 5, 'model__min_samples_leaf': 4, 'model__max_features': 'sqrt', 'model__min_samples_split': 8}"



Best repeated-CV parameters:
{'model__max_depth': 5, 'model__max_features': 0.75, 'model__min_samples_leaf': 6, 'model__min_samples_split': 8}

Repeated-CV gain versus EXP-003: +0.0079

EXP-003 grouped scores:
[0.7486 0.8588 0.8276 0.8245 0.8757]
EXP-003 grouped mean: 0.8270

Best candidate grouped scores:
[0.7371 0.8588 0.8218 0.8298 0.8531]
Best candidate grouped mean: 0.8201

Grouped-CV gain versus EXP-003: -0.0069

Submission decision checks:


,check,result,observed
0,Repeated CV gain >= 0.005,True,0.007855
1,CV stability remains acceptable,True,-0.002193
2,Grouped CV gain >= -0.005,False,-0.006891
3,Parameters differ from baseline,True,NaN



Submission justified: False

Finalist results saved to:
C:\Users\Owner\Documents\Github\machine-learning-lab\00-Kaggle\01-Titanic - Machine Learning from Disaster\results\exp007_rf_finalists.csv

No Kaggle submission recommended. EXP-003 remains the champion.


### EXP-008 — Ticket and Prefix Features


In [27]:
# ============================================================
# EXP-008: RANDOM FOREST WITH TICKET-GROUP FEATURES
# ============================================================

import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import (
    RepeatedStratifiedKFold,
    StratifiedGroupKFold,
    cross_val_score,
)
from sklearn.pipeline import Pipeline


# ------------------------------------------------------------
# 1. CUSTOM FOLD-SAFE TICKET FEATURE ENGINEER
# ------------------------------------------------------------

class TicketFeatureEngineer(BaseEstimator, TransformerMixin):
    """
    Learn ticket frequencies using only the training portion
    of each validation fold.

    Adds:
    - TicketGroupSize
    - SharedTicket
    - TicketPrefix
    """

    @staticmethod
    def clean_ticket(ticket_series):
        return (
            ticket_series
            .fillna("UNKNOWN")
            .astype(str)
            .str.upper()
            .str.replace(r"\s+", " ", regex=True)
            .str.strip()
        )

    @staticmethod
    def extract_prefix(ticket_series):
        prefixes = (
            ticket_series
            .str.replace(r"\d+", "", regex=True)
            .str.replace(r"[^A-Z]+", " ", regex=True)
            .str.strip()
        )

        return prefixes.replace(
            "",
            "NO_PREFIX",
        )

    def fit(self, X, y=None):
        X = X.copy()

        clean_tickets = self.clean_ticket(
            X["Ticket"]
        )

        self.ticket_counts_ = (
            clean_tickets
            .value_counts()
            .to_dict()
        )

        return self

    def transform(self, X):
        X = X.copy()

        clean_tickets = self.clean_ticket(
            X["Ticket"]
        )

        X["TicketGroupSize"] = (
            clean_tickets
            .map(self.ticket_counts_)
            .fillna(1)
            .astype(int)
        )

        X["SharedTicket"] = (
            X["TicketGroupSize"] > 1
        ).astype(int)

        X["TicketPrefix"] = self.extract_prefix(
            clean_tickets
        )

        return X


# ------------------------------------------------------------
# 2. INCLUDE RAW TICKET FOR FEATURE ENGINEERING
# ------------------------------------------------------------

ticket_input_columns = (
    feature_columns
    + ["Ticket"]
)

X_ticket = train_features[
    ticket_input_columns
].copy()

X_test_ticket = test_features[
    ticket_input_columns
].copy()


# ------------------------------------------------------------
# 3. UPDATE FEATURE LISTS
# ------------------------------------------------------------

ticket_numeric_features = (
    numeric_features
    + [
        "TicketGroupSize",
        "SharedTicket",
    ]
)

ticket_categorical_features = (
    categorical_features
    + [
        "TicketPrefix",
    ]
)


# ------------------------------------------------------------
# 4. TICKET-AWARE PREPROCESSOR
# ------------------------------------------------------------

ticket_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_pipeline,
            ticket_numeric_features,
        ),
        (
            "categorical",
            categorical_pipeline,
            ticket_categorical_features,
        ),
    ]
)


# ------------------------------------------------------------
# 5. SAME RANDOM FOREST AS EXP-003
# ------------------------------------------------------------

ticket_rf_model = RandomForestClassifier(
    n_estimators=500,
    max_depth=5,
    min_samples_split=8,
    min_samples_leaf=4,
    max_features="sqrt",
    random_state=42,

    # Avoid nested parallel processing during CV.
    n_jobs=1,
)


ticket_rf_pipeline = Pipeline(
    steps=[
        (
            "ticket_features",
            TicketFeatureEngineer(),
        ),
        (
            "preprocessing",
            ticket_preprocessor,
        ),
        (
            "model",
            ticket_rf_model,
        ),
    ]
)


# ------------------------------------------------------------
# 6. BASELINE PIPELINE FOR FAIR COMPARISON
# ------------------------------------------------------------

baseline_exp008_pipeline = clone(
    random_forest_pipeline
)

baseline_exp008_pipeline.set_params(
    model__n_jobs=1
)


# ------------------------------------------------------------
# 7. REPEATED STRATIFIED VALIDATION
# ------------------------------------------------------------

repeated_cv_exp008 = RepeatedStratifiedKFold(
    n_splits=5,
    n_repeats=5,
    random_state=42,
)


baseline_repeated_scores = cross_val_score(
    baseline_exp008_pipeline,
    X,
    y,
    cv=repeated_cv_exp008,
    scoring="accuracy",
    n_jobs=-1,
)


ticket_repeated_scores = cross_val_score(
    ticket_rf_pipeline,
    X_ticket,
    y,
    cv=repeated_cv_exp008,
    scoring="accuracy",
    n_jobs=-1,
)


baseline_repeated_mean = (
    baseline_repeated_scores.mean()
)

ticket_repeated_mean = (
    ticket_repeated_scores.mean()
)

baseline_repeated_std = (
    baseline_repeated_scores.std()
)

ticket_repeated_std = (
    ticket_repeated_scores.std()
)

repeated_cv_gain = (
    ticket_repeated_mean
    - baseline_repeated_mean
)


print("EXP-003 repeated CV:")
print(
    f"Mean: {baseline_repeated_mean:.4f}"
)

print(
    f"SD:   {baseline_repeated_std:.4f}"
)


print("\nTicket-feature repeated CV:")
print(
    f"Mean: {ticket_repeated_mean:.4f}"
)

print(
    f"SD:   {ticket_repeated_std:.4f}"
)


print(
    "\nRepeated-CV gain:",
    f"{repeated_cv_gain:+.4f}",
)


# ------------------------------------------------------------
# 8. RECREATE FAMILY GROUPS
# ------------------------------------------------------------

validation_data_exp008 = (
    train_features.copy()
)

validation_data_exp008["Surname"] = (
    validation_data_exp008["Name"]
    .str.extract(
        r"^([^,]+),",
        expand=False,
    )
    .str.strip()
    .str.lower()
)


validation_data_exp008["FamilyGroup"] = np.where(
    validation_data_exp008["FamilySize"] > 1,
    (
        validation_data_exp008["Surname"]
        + "_family_"
        + validation_data_exp008[
            "FamilySize"
        ].astype(int).astype(str)
    ),
    (
        "solo_passenger_"
        + validation_data_exp008[
            "PassengerId"
        ].astype(str)
    ),
)


family_groups_exp008 = (
    validation_data_exp008["FamilyGroup"]
)


group_cv_exp008 = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)


# ------------------------------------------------------------
# 9. GROUP-AWARE ROBUSTNESS CHECK
# ------------------------------------------------------------

baseline_group_scores = cross_val_score(
    baseline_exp008_pipeline,
    X,
    y,
    cv=group_cv_exp008,
    groups=family_groups_exp008,
    scoring="accuracy",
    n_jobs=-1,
)


ticket_group_scores = cross_val_score(
    ticket_rf_pipeline,
    X_ticket,
    y,
    cv=group_cv_exp008,
    groups=family_groups_exp008,
    scoring="accuracy",
    n_jobs=-1,
)


baseline_group_mean = (
    baseline_group_scores.mean()
)

ticket_group_mean = (
    ticket_group_scores.mean()
)

grouped_cv_gain = (
    ticket_group_mean
    - baseline_group_mean
)


print("\nEXP-003 grouped scores:")
print(
    baseline_group_scores.round(4)
)

print(
    "EXP-003 grouped mean:",
    f"{baseline_group_mean:.4f}",
)


print("\nTicket-feature grouped scores:")
print(
    ticket_group_scores.round(4)
)

print(
    "Ticket-feature grouped mean:",
    f"{ticket_group_mean:.4f}",
)


print(
    "\nGrouped-CV gain:",
    f"{grouped_cv_gain:+.4f}",
)


# ------------------------------------------------------------
# 10. PREDEFINED SUBMISSION DECISION
# ------------------------------------------------------------

meaningful_repeated_gain = (
    repeated_cv_gain >= 0.003
)

acceptable_grouped_result = (
    grouped_cv_gain >= 0.000
)

acceptable_stability = (
    ticket_repeated_std
    <= baseline_repeated_std + 0.005
)


submission_justified = all(
    [
        meaningful_repeated_gain,
        acceptable_grouped_result,
        acceptable_stability,
    ]
)


decision_summary = pd.DataFrame(
    [
        {
            "check": (
                "Repeated CV gain >= 0.003"
            ),
            "passed": meaningful_repeated_gain,
            "observed": repeated_cv_gain,
        },
        {
            "check": (
                "Grouped CV does not decline"
            ),
            "passed": acceptable_grouped_result,
            "observed": grouped_cv_gain,
        },
        {
            "check": (
                "Repeated CV stability acceptable"
            ),
            "passed": acceptable_stability,
            "observed": (
                ticket_repeated_std
                - baseline_repeated_std
            ),
        },
    ]
)


print("\nSubmission decision:")
display(decision_summary)

print(
    "\nSubmission justified:",
    submission_justified,
)


# ------------------------------------------------------------
# 11. FIT ON ALL TRAINING DATA
# ------------------------------------------------------------

final_ticket_pipeline = clone(
    ticket_rf_pipeline
)

final_ticket_pipeline.set_params(
    model__n_jobs=-1
)

final_ticket_pipeline.fit(
    X_ticket,
    y,
)


exp008_predictions = (
    final_ticket_pipeline
    .predict(X_test_ticket)
    .astype(int)
)


# ------------------------------------------------------------
# 12. COMPARE WITH EXP-003
# ------------------------------------------------------------

exp008_disagreements = (
    exp008_predictions
    != rf_predictions
).sum()


print(
    "\nDiffering predictions versus EXP-003:",
    exp008_disagreements,
)


prediction_comparison_exp008 = pd.DataFrame(
    {
        "PassengerId": test["PassengerId"],
        "exp003_rf": rf_predictions,
        "exp008_ticket_rf": exp008_predictions,
    }
)

display(
    prediction_comparison_exp008[
        prediction_comparison_exp008[
            "exp003_rf"
        ]
        != prediction_comparison_exp008[
            "exp008_ticket_rf"
        ]
    ]
)


# ------------------------------------------------------------
# 13. CREATE SUBMISSION ONLY IF JUSTIFIED
# ------------------------------------------------------------

if submission_justified:
    exp008_submission = pd.DataFrame(
        {
            "PassengerId": (
                test["PassengerId"]
            ),
            "Survived": exp008_predictions,
        }
    )

    exp008_submission_path = (
        output_path
        / "submission_rf_ticket_features_exp008.csv"
    )

    exp008_submission.to_csv(
        exp008_submission_path,
        index=False,
    )

    print("\nSubmission created:")
    print(
        exp008_submission_path
    )

    print("\nPrediction counts:")
    print(
        exp008_submission["Survived"]
        .value_counts()
        .sort_index()
    )

else:
    print(
        "\nNo Kaggle submission recommended. "
        "EXP-003 remains champion."
    )

EXP-003 repeated CV:
Mean: 0.8271
SD:   0.0236

Ticket-feature repeated CV:
Mean: 0.8224
SD:   0.0234

Repeated-CV gain: -0.0047

EXP-003 grouped scores:
[0.7486 0.8588 0.8276 0.8245 0.8757]
EXP-003 grouped mean: 0.8270

Ticket-feature grouped scores:
[0.7314 0.8531 0.8276 0.8245 0.8701]
Ticket-feature grouped mean: 0.8213

Grouped-CV gain: -0.0057

Submission decision:


,check,passed,observed
0,Repeated CV gain >= 0.003,False,-0.004710
1,Grouped CV does not decline,False,-0.005688
2,Repeated CV stability acceptable,True,-0.000189



Submission justified: False

Differing predictions versus EXP-003: 5


,PassengerId,exp003_rf,exp008_ticket_rf
1,893,0,1
18,910,0,1
21,913,1,0
192,1084,1,0
376,1268,0,1



No Kaggle submission recommended. EXP-003 remains champion.


### EXP-009 — Ticket Group Features


In [29]:
# ============================================================
# EXP-009: RANDOM FOREST WITH TICKET GROUP SIZE ONLY
# ============================================================

import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import (
    RepeatedStratifiedKFold,
    StratifiedGroupKFold,
    cross_val_score,
)
from sklearn.pipeline import Pipeline


# ------------------------------------------------------------
# 1. FOLD-SAFE TICKET GROUP FEATURES
# ------------------------------------------------------------

class TicketGroupFeatureEngineer(
    BaseEstimator,
    TransformerMixin,
):
    """
    Learn ticket frequencies using only the training portion
    of each validation fold.

    Adds:
    - TicketGroupSize
    - SharedTicket
    """

    @staticmethod
    def clean_ticket(ticket_series):
        return (
            ticket_series
            .fillna("UNKNOWN")
            .astype(str)
            .str.upper()
            .str.replace(r"\s+", " ", regex=True)
            .str.strip()
        )

    def fit(self, X, y=None):
        clean_tickets = self.clean_ticket(
            X["Ticket"]
        )

        self.ticket_counts_ = (
            clean_tickets
            .value_counts()
            .to_dict()
        )

        return self

    def transform(self, X):
        X = X.copy()

        clean_tickets = self.clean_ticket(
            X["Ticket"]
        )

        # Unseen tickets are treated as solo tickets.
        X["TicketGroupSize"] = (
            clean_tickets
            .map(self.ticket_counts_)
            .fillna(1)
            .astype(int)
        )

        X["SharedTicket"] = (
            X["TicketGroupSize"] > 1
        ).astype(int)

        return X


# ------------------------------------------------------------
# 2. INCLUDE RAW TICKET AS PIPELINE INPUT
# ------------------------------------------------------------

ticket_group_input_columns = (
    feature_columns
    + ["Ticket"]
)

X_ticket_group = train_features[
    ticket_group_input_columns
].copy()

X_test_ticket_group = test_features[
    ticket_group_input_columns
].copy()


# ------------------------------------------------------------
# 3. UPDATE FEATURE LISTS
# ------------------------------------------------------------

ticket_group_numeric_features = (
    numeric_features
    + [
        "TicketGroupSize",
        "SharedTicket",
    ]
)

# No TicketPrefix in this experiment.
ticket_group_categorical_features = (
    categorical_features
)


# ------------------------------------------------------------
# 4. PREPROCESSOR
# ------------------------------------------------------------

ticket_group_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_pipeline,
            ticket_group_numeric_features,
        ),
        (
            "categorical",
            categorical_pipeline,
            ticket_group_categorical_features,
        ),
    ]
)


# ------------------------------------------------------------
# 5. SAME RANDOM FOREST AS EXP-003
# ------------------------------------------------------------

ticket_group_rf_model = RandomForestClassifier(
    n_estimators=500,
    max_depth=5,
    min_samples_split=8,
    min_samples_leaf=4,
    max_features="sqrt",
    random_state=42,

    # Prevent nested parallelism during CV.
    n_jobs=1,
)


ticket_group_pipeline = Pipeline(
    steps=[
        (
            "ticket_group_features",
            TicketGroupFeatureEngineer(),
        ),
        (
            "preprocessing",
            ticket_group_preprocessor,
        ),
        (
            "model",
            ticket_group_rf_model,
        ),
    ]
)


# ------------------------------------------------------------
# 6. EXP-003 BASELINE
# ------------------------------------------------------------

baseline_exp009_pipeline = clone(
    random_forest_pipeline
)

baseline_exp009_pipeline.set_params(
    model__n_jobs=1
)


# ------------------------------------------------------------
# 7. REPEATED STRATIFIED VALIDATION
# ------------------------------------------------------------

repeated_cv_exp009 = RepeatedStratifiedKFold(
    n_splits=5,
    n_repeats=5,
    random_state=42,
)


baseline_repeated_scores_exp009 = cross_val_score(
    baseline_exp009_pipeline,
    X,
    y,
    cv=repeated_cv_exp009,
    scoring="accuracy",
    n_jobs=-1,
)


ticket_group_repeated_scores = cross_val_score(
    ticket_group_pipeline,
    X_ticket_group,
    y,
    cv=repeated_cv_exp009,
    scoring="accuracy",
    n_jobs=-1,
)


baseline_repeated_mean_exp009 = (
    baseline_repeated_scores_exp009.mean()
)

baseline_repeated_std_exp009 = (
    baseline_repeated_scores_exp009.std()
)

ticket_group_repeated_mean = (
    ticket_group_repeated_scores.mean()
)

ticket_group_repeated_std = (
    ticket_group_repeated_scores.std()
)

repeated_cv_gain_exp009 = (
    ticket_group_repeated_mean
    - baseline_repeated_mean_exp009
)


print("EXP-003 repeated CV:")
print(
    f"Mean: {baseline_repeated_mean_exp009:.4f}"
)
print(
    f"SD:   {baseline_repeated_std_exp009:.4f}"
)

print("\nTicket-group repeated CV:")
print(
    f"Mean: {ticket_group_repeated_mean:.4f}"
)
print(
    f"SD:   {ticket_group_repeated_std:.4f}"
)

print(
    "\nRepeated-CV gain:",
    f"{repeated_cv_gain_exp009:+.4f}",
)


# ------------------------------------------------------------
# 8. FAMILY GROUPS FOR ROBUSTNESS VALIDATION
# ------------------------------------------------------------

validation_data_exp009 = (
    train_features.copy()
)

validation_data_exp009["Surname"] = (
    validation_data_exp009["Name"]
    .str.extract(
        r"^([^,]+),",
        expand=False,
    )
    .str.strip()
    .str.lower()
)

validation_data_exp009["FamilyGroup"] = np.where(
    validation_data_exp009["FamilySize"] > 1,
    (
        validation_data_exp009["Surname"]
        + "_family_"
        + validation_data_exp009[
            "FamilySize"
        ].astype(int).astype(str)
    ),
    (
        "solo_passenger_"
        + validation_data_exp009[
            "PassengerId"
        ].astype(str)
    ),
)

family_groups_exp009 = (
    validation_data_exp009["FamilyGroup"]
)

group_cv_exp009 = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)


# ------------------------------------------------------------
# 9. GROUP-AWARE ROBUSTNESS CHECK
# ------------------------------------------------------------

baseline_group_scores_exp009 = cross_val_score(
    baseline_exp009_pipeline,
    X,
    y,
    cv=group_cv_exp009,
    groups=family_groups_exp009,
    scoring="accuracy",
    n_jobs=-1,
)


ticket_group_group_scores = cross_val_score(
    ticket_group_pipeline,
    X_ticket_group,
    y,
    cv=group_cv_exp009,
    groups=family_groups_exp009,
    scoring="accuracy",
    n_jobs=-1,
)


baseline_group_mean_exp009 = (
    baseline_group_scores_exp009.mean()
)

ticket_group_group_mean = (
    ticket_group_group_scores.mean()
)

grouped_cv_gain_exp009 = (
    ticket_group_group_mean
    - baseline_group_mean_exp009
)


print("\nEXP-003 grouped scores:")
print(
    baseline_group_scores_exp009.round(4)
)

print(
    "EXP-003 grouped mean:",
    f"{baseline_group_mean_exp009:.4f}",
)

print("\nTicket-group grouped scores:")
print(
    ticket_group_group_scores.round(4)
)

print(
    "Ticket-group grouped mean:",
    f"{ticket_group_group_mean:.4f}",
)

print(
    "\nGrouped-CV gain:",
    f"{grouped_cv_gain_exp009:+.4f}",
)


# ------------------------------------------------------------
# 10. PREDEFINED SUBMISSION DECISION
# ------------------------------------------------------------

meaningful_repeated_gain_exp009 = (
    repeated_cv_gain_exp009 >= 0.002
)

acceptable_grouped_result_exp009 = (
    grouped_cv_gain_exp009 >= 0.000
)

acceptable_stability_exp009 = (
    ticket_group_repeated_std
    <= baseline_repeated_std_exp009 + 0.005
)


submission_justified_exp009 = all(
    [
        meaningful_repeated_gain_exp009,
        acceptable_grouped_result_exp009,
        acceptable_stability_exp009,
    ]
)


decision_summary_exp009 = pd.DataFrame(
    [
        {
            "check": (
                "Repeated CV gain >= 0.002"
            ),
            "passed": (
                meaningful_repeated_gain_exp009
            ),
            "observed": repeated_cv_gain_exp009,
        },
        {
            "check": (
                "Grouped CV does not decline"
            ),
            "passed": (
                acceptable_grouped_result_exp009
            ),
            "observed": grouped_cv_gain_exp009,
        },
        {
            "check": (
                "Repeated CV stability acceptable"
            ),
            "passed": (
                acceptable_stability_exp009
            ),
            "observed": (
                ticket_group_repeated_std
                - baseline_repeated_std_exp009
            ),
        },
    ]
)


print("\nSubmission decision:")
display(decision_summary_exp009)

print(
    "\nSubmission justified:",
    submission_justified_exp009,
)


# ------------------------------------------------------------
# 11. FIT CANDIDATE ON ALL TRAINING DATA
# ------------------------------------------------------------

final_ticket_group_pipeline = clone(
    ticket_group_pipeline
)

final_ticket_group_pipeline.set_params(
    model__n_jobs=-1
)

final_ticket_group_pipeline.fit(
    X_ticket_group,
    y,
)


exp009_predictions = (
    final_ticket_group_pipeline
    .predict(X_test_ticket_group)
    .astype(int)
)


# ------------------------------------------------------------
# 12. COMPARE WITH EXP-003
# ------------------------------------------------------------

exp009_disagreements = (
    exp009_predictions
    != rf_predictions
).sum()


print(
    "\nDiffering predictions versus EXP-003:",
    exp009_disagreements,
)


prediction_comparison_exp009 = pd.DataFrame(
    {
        "PassengerId": test["PassengerId"],
        "exp003_rf": rf_predictions,
        "exp009_ticket_group_rf": (
            exp009_predictions
        ),
    }
)


display(
    prediction_comparison_exp009[
        prediction_comparison_exp009[
            "exp003_rf"
        ]
        != prediction_comparison_exp009[
            "exp009_ticket_group_rf"
        ]
    ]
)


# ------------------------------------------------------------
# 13. CREATE SUBMISSION ONLY IF JUSTIFIED
# ------------------------------------------------------------

if submission_justified_exp009:
    exp009_submission = pd.DataFrame(
        {
            "PassengerId": (
                test["PassengerId"]
            ),
            "Survived": exp009_predictions,
        }
    )

    exp009_submission_path = (
        output_path
        / "submission_rf_ticket_groups_exp009.csv"
    )

    exp009_submission.to_csv(
        exp009_submission_path,
        index=False,
    )

    print("\nSubmission created:")
    print(
        exp009_submission_path
    )

    print("\nPrediction counts:")
    print(
        exp009_submission["Survived"]
        .value_counts()
        .sort_index()
    )

else:
    print(
        "\nNo Kaggle submission recommended. "
        "EXP-003 remains champion."
    )

EXP-003 repeated CV:
Mean: 0.8271
SD:   0.0236

Ticket-group repeated CV:
Mean: 0.8301
SD:   0.0254

Repeated-CV gain: +0.0029

EXP-003 grouped scores:
[0.7486 0.8588 0.8276 0.8245 0.8757]
EXP-003 grouped mean: 0.8270

Ticket-group grouped scores:
[0.7371 0.8475 0.8276 0.8351 0.8814]
Ticket-group grouped mean: 0.8257

Grouped-CV gain: -0.0013

Submission decision:


,check,passed,observed
0,Repeated CV gain >= 0.002,True,0.002918
1,Grouped CV does not decline,False,-0.001288
2,Repeated CV stability acceptable,True,0.001824



Submission justified: False

Differing predictions versus EXP-003: 3


,PassengerId,exp003_rf,exp009_ticket_group_rf
1,893,0,1
18,910,0,1
376,1268,0,1



No Kaggle submission recommended. EXP-003 remains champion.


### EXP-010 — Random Forest and Logistic Ensemble

In [31]:
# ============================================================
# EXP-010: RANDOM FOREST + LOGISTIC PROBABILITY ENSEMBLE
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.metrics import accuracy_score
from sklearn.model_selection import (
    RepeatedStratifiedKFold,
    StratifiedGroupKFold,
)


# ------------------------------------------------------------
# 1. PREDEFINE THE ENSEMBLE WEIGHTS
# ------------------------------------------------------------

# These represent the Random Forest share.
# Logistic regression receives the remaining weight.

rf_weights = [
    0.50,
    0.60,
    0.70,
    0.80,
    0.90,
]


# ------------------------------------------------------------
# 2. FUNCTION TO EVALUATE BOTH MODELS AND THEIR BLENDS
# ------------------------------------------------------------

def evaluate_probability_blends(
    splitter,
    X_data,
    y_data,
    random_forest_estimator,
    logistic_estimator,
    rf_weight_values,
    groups=None,
):
    """
    Evaluate Random Forest, logistic regression, and fixed
    probability blends across a supplied validation splitter.

    Each validation prediction is produced by models that were
    fitted without the corresponding validation rows.
    """

    model_scores = {
        "random_forest": [],
        "logistic_regression": [],
    }

    for rf_weight in rf_weight_values:
        ensemble_name = (
            f"ensemble_rf_{rf_weight:.2f}"
        )
        model_scores[ensemble_name] = []

    if groups is None:
        split_iterator = splitter.split(
            X_data,
            y_data,
        )
    else:
        split_iterator = splitter.split(
            X_data,
            y_data,
            groups=groups,
        )

    for split_number, (
        train_indices,
        valid_indices,
    ) in enumerate(
        split_iterator,
        start=1,
    ):
        X_train_fold = X_data.iloc[
            train_indices
        ]

        X_valid_fold = X_data.iloc[
            valid_indices
        ]

        y_train_fold = y_data.iloc[
            train_indices
        ]

        y_valid_fold = y_data.iloc[
            valid_indices
        ]

        rf_fold_model = clone(
            random_forest_estimator
        )

        logistic_fold_model = clone(
            logistic_estimator
        )

        # No nested CV here, so the forest may use all cores.
        rf_fold_model.set_params(
            model__n_jobs=-1
        )

        rf_fold_model.fit(
            X_train_fold,
            y_train_fold,
        )

        logistic_fold_model.fit(
            X_train_fold,
            y_train_fold,
        )

        rf_probability = (
            rf_fold_model
            .predict_proba(X_valid_fold)[:, 1]
        )

        logistic_probability = (
            logistic_fold_model
            .predict_proba(X_valid_fold)[:, 1]
        )

        rf_prediction = (
            rf_probability >= 0.50
        ).astype(int)

        logistic_prediction = (
            logistic_probability >= 0.50
        ).astype(int)

        model_scores[
            "random_forest"
        ].append(
            accuracy_score(
                y_valid_fold,
                rf_prediction,
            )
        )

        model_scores[
            "logistic_regression"
        ].append(
            accuracy_score(
                y_valid_fold,
                logistic_prediction,
            )
        )

        for rf_weight in rf_weight_values:
            logistic_weight = (
                1.0 - rf_weight
            )

            ensemble_probability = (
                rf_weight
                * rf_probability
                + logistic_weight
                * logistic_probability
            )

            ensemble_prediction = (
                ensemble_probability >= 0.50
            ).astype(int)

            ensemble_name = (
                f"ensemble_rf_{rf_weight:.2f}"
            )

            model_scores[
                ensemble_name
            ].append(
                accuracy_score(
                    y_valid_fold,
                    ensemble_prediction,
                )
            )

    result_rows = []

    for model_name, scores in model_scores.items():
        scores = np.asarray(
            scores,
            dtype=float,
        )

        if model_name.startswith(
            "ensemble_rf_"
        ):
            rf_weight = float(
                model_name.split("_")[-1]
            )

            model_type = "ensemble"
            logistic_weight = (
                1.0 - rf_weight
            )

        elif model_name == "random_forest":
            model_type = "baseline"
            rf_weight = 1.0
            logistic_weight = 0.0

        else:
            model_type = "baseline"
            rf_weight = 0.0
            logistic_weight = 1.0

        result_rows.append(
            {
                "model": model_name,
                "model_type": model_type,
                "rf_weight": rf_weight,
                "logistic_weight": (
                    logistic_weight
                ),
                "mean_accuracy": scores.mean(),
                "std_accuracy": scores.std(),
                "minimum_accuracy": scores.min(),
                "maximum_accuracy": scores.max(),
                "number_of_scores": len(scores),
                "scores": scores,
            }
        )

    results = pd.DataFrame(
        result_rows
    ).sort_values(
        [
            "mean_accuracy",
            "std_accuracy",
        ],
        ascending=[
            False,
            True,
        ],
    ).reset_index(drop=True)

    return results


# ------------------------------------------------------------
# 3. REPEATED STRATIFIED VALIDATION
# ------------------------------------------------------------

repeated_cv_exp010 = RepeatedStratifiedKFold(
    n_splits=5,
    n_repeats=5,
    random_state=42,
)


repeated_results_exp010 = (
    evaluate_probability_blends(
        splitter=repeated_cv_exp010,
        X_data=X,
        y_data=y,
        random_forest_estimator=(
            random_forest_pipeline
        ),
        logistic_estimator=(
            logistic_pipeline
        ),
        rf_weight_values=rf_weights,
    )
)


print(
    "Repeated stratified validation:"
)

display(
    repeated_results_exp010[
        [
            "model",
            "rf_weight",
            "logistic_weight",
            "mean_accuracy",
            "std_accuracy",
            "minimum_accuracy",
            "maximum_accuracy",
        ]
    ].style.format(
        {
            "rf_weight": "{:.2f}",
            "logistic_weight": "{:.2f}",
            "mean_accuracy": "{:.4f}",
            "std_accuracy": "{:.4f}",
            "minimum_accuracy": "{:.4f}",
            "maximum_accuracy": "{:.4f}",
        }
    )
)


# ------------------------------------------------------------
# 4. SELECT THE BEST ACTUAL ENSEMBLE
# ------------------------------------------------------------

ensemble_results_exp010 = (
    repeated_results_exp010[
        repeated_results_exp010[
            "model_type"
        ] == "ensemble"
    ]
    .sort_values(
        [
            "mean_accuracy",
            "std_accuracy",
        ],
        ascending=[
            False,
            True,
        ],
    )
    .reset_index(drop=True)
)


best_ensemble_exp010 = (
    ensemble_results_exp010.iloc[0]
)


best_rf_weight_exp010 = float(
    best_ensemble_exp010["rf_weight"]
)

best_logistic_weight_exp010 = (
    1.0 - best_rf_weight_exp010
)


rf_repeated_result_exp010 = (
    repeated_results_exp010[
        repeated_results_exp010[
            "model"
        ] == "random_forest"
    ]
    .iloc[0]
)


logistic_repeated_result_exp010 = (
    repeated_results_exp010[
        repeated_results_exp010[
            "model"
        ] == "logistic_regression"
    ]
    .iloc[0]
)


rf_repeated_mean_exp010 = float(
    rf_repeated_result_exp010[
        "mean_accuracy"
    ]
)

rf_repeated_std_exp010 = float(
    rf_repeated_result_exp010[
        "std_accuracy"
    ]
)

ensemble_repeated_mean_exp010 = float(
    best_ensemble_exp010[
        "mean_accuracy"
    ]
)

ensemble_repeated_std_exp010 = float(
    best_ensemble_exp010[
        "std_accuracy"
    ]
)


repeated_gain_exp010 = (
    ensemble_repeated_mean_exp010
    - rf_repeated_mean_exp010
)


print(
    "\nSelected Random Forest weight:",
    f"{best_rf_weight_exp010:.2f}",
)

print(
    "Selected logistic weight:",
    f"{best_logistic_weight_exp010:.2f}",
)

print(
    "\nEXP-003 repeated-CV mean:",
    f"{rf_repeated_mean_exp010:.4f}",
)

print(
    "EXP-003 repeated-CV SD:",
    f"{rf_repeated_std_exp010:.4f}",
)

print(
    "\nBest ensemble repeated-CV mean:",
    f"{ensemble_repeated_mean_exp010:.4f}",
)

print(
    "Best ensemble repeated-CV SD:",
    f"{ensemble_repeated_std_exp010:.4f}",
)

print(
    "\nRepeated-CV gain:",
    f"{repeated_gain_exp010:+.4f}",
)


# ------------------------------------------------------------
# 5. RECREATE FAMILY GROUPS
# ------------------------------------------------------------

validation_data_exp010 = (
    train_features.copy()
)


validation_data_exp010["Surname"] = (
    validation_data_exp010["Name"]
    .str.extract(
        r"^([^,]+),",
        expand=False,
    )
    .str.strip()
    .str.lower()
)


validation_data_exp010[
    "FamilyGroup"
] = np.where(
    validation_data_exp010[
        "FamilySize"
    ] > 1,
    (
        validation_data_exp010["Surname"]
        + "_family_"
        + validation_data_exp010[
            "FamilySize"
        ].astype(int).astype(str)
    ),
    (
        "solo_passenger_"
        + validation_data_exp010[
            "PassengerId"
        ].astype(str)
    ),
)


family_groups_exp010 = (
    validation_data_exp010[
        "FamilyGroup"
    ]
)


# ------------------------------------------------------------
# 6. GROUP-AWARE ROBUSTNESS VALIDATION
# ------------------------------------------------------------

group_cv_exp010 = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)


grouped_results_exp010 = (
    evaluate_probability_blends(
        splitter=group_cv_exp010,
        X_data=X,
        y_data=y,
        random_forest_estimator=(
            random_forest_pipeline
        ),
        logistic_estimator=(
            logistic_pipeline
        ),
        rf_weight_values=rf_weights,
        groups=family_groups_exp010,
    )
)


print(
    "\nGroup-aware validation:"
)

display(
    grouped_results_exp010[
        [
            "model",
            "rf_weight",
            "logistic_weight",
            "mean_accuracy",
            "std_accuracy",
            "minimum_accuracy",
            "maximum_accuracy",
        ]
    ].style.format(
        {
            "rf_weight": "{:.2f}",
            "logistic_weight": "{:.2f}",
            "mean_accuracy": "{:.4f}",
            "std_accuracy": "{:.4f}",
            "minimum_accuracy": "{:.4f}",
            "maximum_accuracy": "{:.4f}",
        }
    )
)


rf_group_result_exp010 = (
    grouped_results_exp010[
        grouped_results_exp010[
            "model"
        ] == "random_forest"
    ]
    .iloc[0]
)


selected_group_result_exp010 = (
    grouped_results_exp010[
        np.isclose(
            grouped_results_exp010[
                "rf_weight"
            ],
            best_rf_weight_exp010,
        )
        & (
            grouped_results_exp010[
                "model_type"
            ] == "ensemble"
        )
    ]
    .iloc[0]
)


rf_group_mean_exp010 = float(
    rf_group_result_exp010[
        "mean_accuracy"
    ]
)

ensemble_group_mean_exp010 = float(
    selected_group_result_exp010[
        "mean_accuracy"
    ]
)


grouped_gain_exp010 = (
    ensemble_group_mean_exp010
    - rf_group_mean_exp010
)


print(
    "\nEXP-003 grouped mean:",
    f"{rf_group_mean_exp010:.4f}",
)

print(
    "Selected ensemble grouped mean:",
    f"{ensemble_group_mean_exp010:.4f}",
)

print(
    "\nGrouped-CV gain:",
    f"{grouped_gain_exp010:+.4f}",
)


# ------------------------------------------------------------
# 7. PREDEFINED SUBMISSION DECISION
# ------------------------------------------------------------

meaningful_repeated_gain_exp010 = (
    repeated_gain_exp010 >= 0.003
)


acceptable_grouped_result_exp010 = (
    grouped_gain_exp010 >= 0.000
)


acceptable_stability_exp010 = (
    ensemble_repeated_std_exp010
    <= rf_repeated_std_exp010 + 0.005
)


genuine_blend_exp010 = (
    0.0 < best_rf_weight_exp010 < 1.0
)


submission_justified_exp010 = all(
    [
        meaningful_repeated_gain_exp010,
        acceptable_grouped_result_exp010,
        acceptable_stability_exp010,
        genuine_blend_exp010,
    ]
)


decision_summary_exp010 = pd.DataFrame(
    [
        {
            "check": (
                "Repeated CV gain >= 0.003"
            ),
            "passed": (
                meaningful_repeated_gain_exp010
            ),
            "observed": repeated_gain_exp010,
        },
        {
            "check": (
                "Grouped CV does not decline"
            ),
            "passed": (
                acceptable_grouped_result_exp010
            ),
            "observed": grouped_gain_exp010,
        },
        {
            "check": (
                "Repeated CV stability acceptable"
            ),
            "passed": (
                acceptable_stability_exp010
            ),
            "observed": (
                ensemble_repeated_std_exp010
                - rf_repeated_std_exp010
            ),
        },
        {
            "check": (
                "Selected candidate uses both models"
            ),
            "passed": genuine_blend_exp010,
            "observed": (
                best_rf_weight_exp010
            ),
        },
    ]
)


print(
    "\nSubmission decision:"
)

display(
    decision_summary_exp010
)

print(
    "\nSubmission justified:",
    submission_justified_exp010,
)


# ------------------------------------------------------------
# 8. FIT BOTH MODELS ON ALL TRAINING DATA
# ------------------------------------------------------------

final_rf_exp010 = clone(
    random_forest_pipeline
)

final_rf_exp010.set_params(
    model__n_jobs=-1
)


final_logistic_exp010 = clone(
    logistic_pipeline
)


final_rf_exp010.fit(
    X,
    y,
)

final_logistic_exp010.fit(
    X,
    y,
)


rf_test_probability_exp010 = (
    final_rf_exp010
    .predict_proba(X_test)[:, 1]
)


logistic_test_probability_exp010 = (
    final_logistic_exp010
    .predict_proba(X_test)[:, 1]
)


ensemble_test_probability_exp010 = (
    best_rf_weight_exp010
    * rf_test_probability_exp010
    + best_logistic_weight_exp010
    * logistic_test_probability_exp010
)


exp010_predictions = (
    ensemble_test_probability_exp010
    >= 0.50
).astype(int)


# ------------------------------------------------------------
# 9. COMPARE WITH EXP-003
# ------------------------------------------------------------

exp010_disagreements = (
    exp010_predictions
    != rf_predictions
).sum()


print(
    "\nDiffering predictions versus EXP-003:",
    exp010_disagreements,
)


prediction_comparison_exp010 = pd.DataFrame(
    {
        "PassengerId": test["PassengerId"],
        "exp003_rf": rf_predictions,
        "exp010_ensemble": (
            exp010_predictions
        ),
        "rf_probability": (
            rf_test_probability_exp010
        ),
        "logistic_probability": (
            logistic_test_probability_exp010
        ),
        "ensemble_probability": (
            ensemble_test_probability_exp010
        ),
    }
)


display(
    prediction_comparison_exp010[
        prediction_comparison_exp010[
            "exp003_rf"
        ]
        != prediction_comparison_exp010[
            "exp010_ensemble"
        ]
    ].round(
        {
            "rf_probability": 3,
            "logistic_probability": 3,
            "ensemble_probability": 3,
        }
    )
)


# ------------------------------------------------------------
# 10. SAVE VALIDATION RESULTS
# ------------------------------------------------------------

results_directory = (
    output_path / "results"
)

results_directory.mkdir(
    parents=True,
    exist_ok=True,
)


repeated_results_path_exp010 = (
    results_directory
    / "exp010_repeated_blend_results.csv"
)


grouped_results_path_exp010 = (
    results_directory
    / "exp010_grouped_blend_results.csv"
)


repeated_results_exp010.drop(
    columns=["scores"]
).to_csv(
    repeated_results_path_exp010,
    index=False,
)


grouped_results_exp010.drop(
    columns=["scores"]
).to_csv(
    grouped_results_path_exp010,
    index=False,
)


print(
    "\nRepeated results saved to:"
)

print(
    repeated_results_path_exp010
)


print(
    "\nGrouped results saved to:"
)

print(
    grouped_results_path_exp010
)


# ------------------------------------------------------------
# 11. CREATE SUBMISSION ONLY IF JUSTIFIED
# ------------------------------------------------------------

if submission_justified_exp010:
    exp010_submission = pd.DataFrame(
        {
            "PassengerId": (
                test["PassengerId"]
            ),
            "Survived": exp010_predictions,
        }
    )

    exp010_submission_path = (
        output_path
        / "submission_rf_logistic_ensemble_exp010.csv"
    )

    exp010_submission.to_csv(
        exp010_submission_path,
        index=False,
    )

    print(
        "\nSubmission created:"
    )

    print(
        exp010_submission_path
    )

    print(
        "\nPrediction counts:"
    )

    print(
        exp010_submission[
            "Survived"
        ]
        .value_counts()
        .sort_index()
    )

else:
    print(
        "\nNo Kaggle submission recommended. "
        "EXP-003 remains champion."
    )

Repeated stratified validation:


,model,rf_weight,logistic_weight,mean_accuracy,std_accuracy,minimum_accuracy,maximum_accuracy
0,ensemble_rf_0.60,0.60,0.40,0.8330,0.0231,0.7921,0.8715
1,ensemble_rf_0.70,0.70,0.30,0.8316,0.0233,0.7921,0.8708
2,ensemble_rf_0.50,0.50,0.50,0.8316,0.0234,0.7865,0.8708
3,ensemble_rf_0.80,0.80,0.20,0.8283,0.0234,0.7809,0.8764
4,ensemble_rf_0.90,0.90,0.10,0.8278,0.0248,0.7809,0.8820
5,random_forest,1.00,0.00,0.8271,0.0236,0.7809,0.8764
6,logistic_regression,0.00,1.00,0.8260,0.0241,0.7753,0.8659



Selected Random Forest weight: 0.60
Selected logistic weight: 0.40

EXP-003 repeated-CV mean: 0.8271
EXP-003 repeated-CV SD: 0.0236

Best ensemble repeated-CV mean: 0.8330
Best ensemble repeated-CV SD: 0.0231

Repeated-CV gain: +0.0058

Group-aware validation:


,model,rf_weight,logistic_weight,mean_accuracy,std_accuracy,minimum_accuracy,maximum_accuracy
0,ensemble_rf_0.50,0.50,0.50,0.8349,0.0406,0.7657,0.8814
1,ensemble_rf_0.60,0.60,0.40,0.8348,0.0409,0.7657,0.8814
2,ensemble_rf_0.70,0.70,0.30,0.8348,0.0458,0.7543,0.8870
3,ensemble_rf_0.80,0.80,0.20,0.8314,0.0455,0.7486,0.8757
4,logistic_regression,0.00,1.00,0.8295,0.0404,0.7600,0.8757
5,ensemble_rf_0.90,0.90,0.10,0.8271,0.0445,0.7486,0.8701
6,random_forest,1.00,0.00,0.8270,0.0437,0.7486,0.8757



EXP-003 grouped mean: 0.8270
Selected ensemble grouped mean: 0.8348

Grouped-CV gain: +0.0078

Submission decision:


,check,passed,observed
0,Repeated CV gain >= 0.003,True,0.005836
1,Grouped CV does not decline,True,0.007797
2,Repeated CV stability acceptable,True,-0.000436
3,Selected candidate uses both models,True,0.600000



Submission justified: True

Differing predictions versus EXP-003: 6


,PassengerId,exp003_rf,exp010_ensemble,rf_probability,logistic_probability,ensemble_probability
1,893,0,1,0.500,0.560,0.524
18,910,0,1,0.488,0.542,0.509
73,965,0,1,0.417,0.639,0.505
181,1073,0,1,0.462,0.558,0.500
293,1185,1,0,0.517,0.294,0.428
339,1231,0,1,0.423,0.712,0.539



Repeated results saved to:
C:\Users\Owner\Documents\Github\machine-learning-lab\00-Kaggle\01-Titanic - Machine Learning from Disaster\results\exp010_repeated_blend_results.csv

Grouped results saved to:
C:\Users\Owner\Documents\Github\machine-learning-lab\00-Kaggle\01-Titanic - Machine Learning from Disaster\results\exp010_grouped_blend_results.csv

Submission created:
C:\Users\Owner\Documents\Github\machine-learning-lab\00-Kaggle\01-Titanic - Machine Learning from Disaster\submission_rf_logistic_ensemble_exp010.csv

Prediction counts:
Survived
0    255
1    163
Name: count, dtype: int64


## Final Conclusions

The strongest model was a constrained Random Forest with a Kaggle accuracy of **0.78708**.

XGBoost achieved stronger initial cross-validation but transferred poorly to the hidden test set. Logistic regression generalized slightly better, while Random Forest captured useful nonlinear interactions without becoming as fragile as the boosted model.

Later experiments involving interaction features, grouped age imputation, hyperparameter tuning, ticket features, and probability ensembling did not reliably improve the champion.

The project was stopped once additional experiments became more focused on changing a handful of leaderboard predictions than on generating transferable machine-learning insight.


### EXP-011 — Engineered Feature Ablation

**Hypothesis:** Some engineered features in the champion model may be redundant or harmful.

**Constant:** Random Forest parameters, preprocessing approach, metric, random seed, and validation strategy.

**Change:** Remove one engineered feature group at a time.

**Success criterion:** An ablated version must improve repeated CV without meaningfully weakening group-aware CV.

**Kaggle submission:** Not required unless a version produces a convincing local improvement.

In [4]:
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.model_selection import (
    RepeatedStratifiedKFold,
    StratifiedGroupKFold,
    cross_val_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder


# ============================================================
# 1. FEATURE VERSIONS TO TEST
# ============================================================

raw_features = [
    "Pclass",
    "Sex",
    "Age",
    "SibSp",
    "Parch",
    "Fare",
    "Embarked",
]

engineered_features = [
    "Title",
    "FamilySize",
    "IsAlone",
    "Deck",
    "FarePerPerson",
]

full_features = raw_features + engineered_features

feature_variants = {
    "Full champion": full_features,

    "No Title": [
        feature
        for feature in full_features
        if feature != "Title"
    ],

    "No family engineering": [
        feature
        for feature in full_features
        if feature not in ["FamilySize", "IsAlone"]
    ],

    "No Deck": [
        feature
        for feature in full_features
        if feature != "Deck"
    ],

    "No FarePerPerson": [
        feature
        for feature in full_features
        if feature != "FarePerPerson"
    ],

    "Raw features only": raw_features,
}


# ============================================================
# 2. CREATE FAMILY GROUPS
# ============================================================

group_data = train.copy()

group_data["Surname"] = (
    group_data["Name"]
    .str.split(",")
    .str[0]
    .str.strip()
)

group_data["CalculatedFamilySize"] = (
    group_data["SibSp"]
    + group_data["Parch"]
    + 1
)

group_data["FamilyGroup"] = np.where(
    group_data["CalculatedFamilySize"] > 1,
    (
        group_data["Surname"]
        + "_"
        + group_data["CalculatedFamilySize"].astype(str)
    ),
    "Solo_" + group_data["PassengerId"].astype(str),
)

family_groups_exp011 = group_data["FamilyGroup"]


# ============================================================
# 3. BUILD THE CHAMPION MODEL
# ============================================================

def build_exp011_pipeline(features):
    categorical_candidates = {
        "Pclass",
        "Sex",
        "Embarked",
        "Title",
        "Deck",
    }

    categorical_features_exp011 = [
        feature
        for feature in features
        if feature in categorical_candidates
    ]

    numeric_features_exp011 = [
        feature
        for feature in features
        if feature not in categorical_candidates
    ]

    numeric_pipeline_exp011 = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
        ]
    )

    categorical_pipeline_exp011 = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(strategy="most_frequent"),
            ),
            (
                "onehot",
                OneHotEncoder(handle_unknown="ignore"),
            ),
        ]
    )

    preprocessor_exp011 = ColumnTransformer(
        transformers=[
            (
                "numeric",
                numeric_pipeline_exp011,
                numeric_features_exp011,
            ),
            (
                "categorical",
                categorical_pipeline_exp011,
                categorical_features_exp011,
            ),
        ]
    )

    model_exp011 = RandomForestClassifier(
        n_estimators=500,
        max_depth=5,
        min_samples_split=8,
        min_samples_leaf=4,
        max_features="sqrt",
        random_state=42,
        n_jobs=-1,
    )

    return Pipeline(
        steps=[
            ("preprocessing", preprocessor_exp011),
            ("model", model_exp011),
        ]
    )


# ============================================================
# 4. DEFINE VALIDATION
# ============================================================

repeated_cv_exp011 = RepeatedStratifiedKFold(
    n_splits=5,
    n_repeats=5,
    random_state=42,
)

group_cv_exp011 = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)


# ============================================================
# 5. RUN EACH FEATURE VERSION
# ============================================================

variant_scores_exp011 = {}
results_exp011 = []

for variant_name, features in feature_variants.items():
    pipeline_exp011 = build_exp011_pipeline(features)

    X_variant = train_features[features]

    repeated_scores = cross_val_score(
        pipeline_exp011,
        X_variant,
        y,
        cv=repeated_cv_exp011,
        scoring="accuracy",
        n_jobs=-1,
    )

    grouped_scores = cross_val_score(
        pipeline_exp011,
        X_variant,
        y,
        cv=group_cv_exp011,
        groups=family_groups_exp011,
        scoring="accuracy",
        n_jobs=-1,
    )

    variant_scores_exp011[variant_name] = repeated_scores

    results_exp011.append(
        {
            "variant": variant_name,
            "repeated_cv_mean": repeated_scores.mean(),
            "repeated_cv_std": repeated_scores.std(),
            "group_cv_mean": grouped_scores.mean(),
            "group_cv_std": grouped_scores.std(),
        }
    )


# ============================================================
# 6. COMPARE WITH THE FULL CHAMPION
# ============================================================

full_scores_exp011 = variant_scores_exp011["Full champion"]
full_mean_exp011 = full_scores_exp011.mean()

results_exp011_df = pd.DataFrame(results_exp011)

results_exp011_df["change_vs_full"] = (
    results_exp011_df["repeated_cv_mean"]
    - full_mean_exp011
)

full_win_rates = {}

for variant_name, scores in variant_scores_exp011.items():
    if variant_name == "Full champion":
        full_win_rates[variant_name] = np.nan
    else:
        full_win_rates[variant_name] = (
            full_scores_exp011 > scores
        ).mean()

results_exp011_df["full_model_win_rate"] = (
    results_exp011_df["variant"].map(full_win_rates)
)

results_exp011_df = (
    results_exp011_df
    .sort_values("repeated_cv_mean", ascending=False)
    .reset_index(drop=True)
)

display(results_exp011_df.round(4))

,variant,repeated_cv_mean,repeated_cv_std,group_cv_mean,group_cv_std,change_vs_full,full_model_win_rate
0,No FarePerPerson,0.8298,0.0233,0.8246,0.0167,0.0027,0.20
1,No Deck,0.8292,0.0211,0.8189,0.0155,0.0020,0.24
2,Full champion,0.8271,0.0236,0.8246,0.0176,0.0000,NaN
3,No family engineering,0.8240,0.0237,0.8221,0.0191,-0.0032,0.60
4,Raw features only,0.8173,0.0217,0.8166,0.0213,-0.0099,0.76
5,No Title,0.8121,0.0256,0.8053,0.0210,-0.0150,0.76


### EXP-012

In [7]:
import numpy as np
import pandas as pd


# Extract surname from Name.
wcg_data = train.copy()

wcg_data["Surname"] = (
    wcg_data["Name"]
    .str.split(",")
    .str[0]
    .str.strip()
)

# Match the family-size definition already used in the notebook.
wcg_data["FamilySize"] = (
    wcg_data["SibSp"]
    + wcg_data["Parch"]
    + 1
)

# Surname alone can combine unrelated people.
# Adding family size makes the family key more specific.
wcg_data["FamilyGroup"] = (
    wcg_data["Surname"]
    + "_"
    + wcg_data["FamilySize"].astype(str)
)

# Use the name title to identify boys.
wcg_data["Title"] = (
    wcg_data["Name"]
    .str.extract(r",\s*([^.]*)\.", expand=False)
    .str.strip()
)

wcg_data["PassengerType"] = np.select(
    [
        wcg_data["Sex"].eq("female"),
        wcg_data["Title"].eq("Master"),
    ],
    [
        "Woman",
        "Boy",
    ],
    default="Man",
)

# Chris's main relational signal focuses on women and boys.
woman_child_data = wcg_data[
    wcg_data["PassengerType"].isin(["Woman", "Boy"])
].copy()

group_summary = (
    woman_child_data
    .groupby("FamilyGroup")
    .agg(
        group_members=("PassengerId", "size"),
        survival_rate=("Survived", "mean"),
        unique_outcomes=("Survived", "nunique"),
        passenger_ids=("PassengerId", list),
    )
    .query("group_members >= 2")
    .sort_values(
        ["unique_outcomes", "group_members"],
        ascending=[True, False],
    )
)

consistent_groups = group_summary[
    group_summary["unique_outcomes"] == 1
]

print("Woman-child groups with at least 2 labelled members:")
print(len(group_summary))

print("\nGroups where everyone shared the same outcome:")
print(len(consistent_groups))

if len(group_summary) > 0:
    consistency_rate = (
        len(consistent_groups) / len(group_summary)
    )

    print(
        "\nGroup consistency rate:",
        round(consistency_rate, 4),
    )

display(group_summary.head(20))

Woman-child groups with at least 2 labelled members:
50

Groups where everyone shared the same outcome:
47

Group consistency rate: 0.94


,group_members,survival_rate,unique_outcomes,passenger_ids
FamilyGroup,,,,
Goodwin_8,5,0.0,1,"[60, 72, 387, 481, 679]"
Rice_6,5,0.0,1,"[17, 172, 279, 788, 886]"
Skoog_6,5,0.0,1,"[64, 168, 635, 643, 820]"
Baclini_4,4,1.0,1,"[449, 470, 645, 859]"
Lefebre_5,4,0.0,1,"[177, 230, 410, 486]"
Palsson_5,4,0.0,1,"[8, 25, 375, 568]"
Panula_6,4,0.0,1,"[51, 165, 639, 825]"
Sage_11,4,0.0,1,"[160, 181, 793, 864]"
Carter_4,3,1.0,1,"[436, 764, 803]"


In [13]:
from sklearn.base import clone
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.metrics import accuracy_score


# ============================================================
# 1. PREPARE RELATIONAL INFORMATION
# ============================================================

relational_data = train.copy()

relational_data["Surname"] = (
    relational_data["Name"]
    .str.split(",")
    .str[0]
    .str.strip()
)

relational_data["CalculatedFamilySize"] = (
    relational_data["SibSp"]
    + relational_data["Parch"]
    + 1
)

relational_data["FamilyGroup"] = (
    relational_data["Surname"]
    + "_"
    + relational_data["CalculatedFamilySize"].astype(str)
)

relational_data["ExtractedTitle"] = (
    relational_data["Name"]
    .str.extract(r",\s*([^.]*)\.", expand=False)
    .str.strip()
)

relational_data["PassengerType"] = np.select(
    [
        relational_data["Sex"].eq("female"),
        relational_data["ExtractedTitle"].eq("Master"),
    ],
    [
        "Woman",
        "Boy",
    ],
    default="Man",
)


# ============================================================
# 2. CHAMPION MODEL
# ============================================================

champion_exp012 = build_exp011_pipeline(
    full_features
)


# ============================================================
# 3. REPEATED VALIDATION
# ============================================================

cv_exp012 = RepeatedStratifiedKFold(
    n_splits=5,
    n_repeats=5,
    random_state=42,
)

fold_results_exp012 = []
changed_rows_exp012 = []

for fold_number, (train_idx, valid_idx) in enumerate(
    cv_exp012.split(train_features, y),
    start=1,
):
    X_train_fold = train_features.iloc[train_idx]
    X_valid_fold = train_features.iloc[valid_idx]

    y_train_fold = y.iloc[train_idx]
    y_valid_fold = y.iloc[valid_idx]

    relation_train = relational_data.iloc[train_idx].copy()
    relation_valid = relational_data.iloc[valid_idx].copy()

    # --------------------------------------------------------
    # Train and predict with the unchanged champion
    # --------------------------------------------------------

    fold_model = clone(champion_exp012)

    fold_model.fit(
        X_train_fold,
        y_train_fold,
    )

    baseline_predictions = fold_model.predict(
        X_valid_fold
    )

    override_predictions = baseline_predictions.copy()

    # --------------------------------------------------------
    # Learn woman-child family outcomes from training fold only
    # --------------------------------------------------------

    # Attach each passenger's target before filtering.
    relation_train["FoldTarget"] = y_train_fold.to_numpy()
    
    # Then keep only women and boys.
    training_women_children = relation_train[
        relation_train["PassengerType"].isin(
            ["Woman", "Boy"]
        )
    ].copy()

    family_outcomes = (
        training_women_children
        .groupby("FamilyGroup")["FoldTarget"]
        .agg(["mean", "count", "nunique"])
    )

    # Only use groups with a completely consistent known outcome.
    minimum_known_labels_exp012 = 2

    reliable_family_outcomes = family_outcomes[
    (family_outcomes["nunique"] == 1)
    & (
        family_outcomes["count"]
        >= minimum_known_labels_exp012
    )
]

    family_survival_map = (
        reliable_family_outcomes["mean"]
        .to_dict()
    )

    family_count_map = (
        reliable_family_outcomes["count"]
        .to_dict()
    )

    changed_in_fold = 0
    correct_before = 0
    correct_after = 0

    # --------------------------------------------------------
    # Apply targeted Chris-style overrides
    # --------------------------------------------------------

    for position, (_, passenger) in enumerate(
        relation_valid.iterrows()
    ):
        family_group = passenger["FamilyGroup"]
        passenger_type = passenger["PassengerType"]

        if family_group not in family_survival_map:
            continue

        known_family_outcome = family_survival_map[
            family_group
        ]

        original_prediction = int(
            baseline_predictions[position]
        )

        new_prediction = original_prediction
        override_reason = None

        # Woman exception:
        # Predict death if all known women/boys in her group died.
        if (
            passenger_type == "Woman"
            and known_family_outcome == 0
        ):
            new_prediction = 0
            override_reason = "Woman in all-death WCG"

        # Boy exception:
        # Predict survival if all known women/boys survived.
        elif (
            passenger_type == "Boy"
            and known_family_outcome == 1
        ):
            new_prediction = 1
            override_reason = "Boy in all-survival WCG"

        if new_prediction != original_prediction:
            override_predictions[position] = new_prediction

            actual = int(y_valid_fold.iloc[position])

            changed_in_fold += 1
            correct_before += int(
                original_prediction == actual
            )
            correct_after += int(
                new_prediction == actual
            )

            changed_rows_exp012.append(
                {
                    "fold": fold_number,
                    "PassengerId": passenger["PassengerId"],
                    "Name": passenger["Name"],
                    "PassengerType": passenger_type,
                    "FamilyGroup": family_group,
                    "known_family_outcome": (
                        known_family_outcome
                    ),
                    "known_label_count": (
                        family_count_map[family_group]
                    ),
                    "original_prediction": (
                        original_prediction
                    ),
                    "override_prediction": (
                        new_prediction
                    ),
                    "actual": actual,
                    "override_reason": override_reason,
                }
            )

    baseline_accuracy = accuracy_score(
        y_valid_fold,
        baseline_predictions,
    )

    override_accuracy = accuracy_score(
        y_valid_fold,
        override_predictions,
    )

    fold_results_exp012.append(
        {
            "fold": fold_number,
            "baseline_accuracy": baseline_accuracy,
            "override_accuracy": override_accuracy,
            "accuracy_change": (
                override_accuracy - baseline_accuracy
            ),
            "predictions_changed": changed_in_fold,
            "changed_accuracy_before": (
                correct_before / changed_in_fold
                if changed_in_fold > 0
                else np.nan
            ),
            "changed_accuracy_after": (
                correct_after / changed_in_fold
                if changed_in_fold > 0
                else np.nan
            ),
        }
    )


# ============================================================
# 4. SUMMARIZE RESULTS
# ============================================================

fold_results_exp012_df = pd.DataFrame(
    fold_results_exp012
)

changed_rows_exp012_df = pd.DataFrame(
    changed_rows_exp012
)

summary_exp012 = pd.DataFrame(
    {
        "metric": [
            "Baseline mean accuracy",
            "Override mean accuracy",
            "Mean accuracy change",
            "Folds improved",
            "Folds worsened",
            "Total prediction changes",
            "Changed-row accuracy before",
            "Changed-row accuracy after",
        ],
        "value": [
            fold_results_exp012_df[
                "baseline_accuracy"
            ].mean(),

            fold_results_exp012_df[
                "override_accuracy"
            ].mean(),

            fold_results_exp012_df[
                "accuracy_change"
            ].mean(),

            (
                fold_results_exp012_df[
                    "accuracy_change"
                ] > 0
            ).sum(),

            (
                fold_results_exp012_df[
                    "accuracy_change"
                ] < 0
            ).sum(),

            len(changed_rows_exp012_df),

            (
                changed_rows_exp012_df[
                    "original_prediction"
                ]
                == changed_rows_exp012_df["actual"]
            ).mean()
            if len(changed_rows_exp012_df) > 0
            else np.nan,

            (
                changed_rows_exp012_df[
                    "override_prediction"
                ]
                == changed_rows_exp012_df["actual"]
            ).mean()
            if len(changed_rows_exp012_df) > 0
            else np.nan,
        ],
    }
)

display(summary_exp012.round(4))
display(fold_results_exp012_df.round(4))

if len(changed_rows_exp012_df) > 0:
    display(
        changed_rows_exp012_df.head(30)
    )

,metric,value
0,Baseline mean accuracy,0.8271
1,Override mean accuracy,0.8260
2,Mean accuracy change,-0.0011
3,Folds improved,0.0000
4,Folds worsened,5.0000
5,Total prediction changes,5.0000
6,Changed-row accuracy before,1.0000
7,Changed-row accuracy after,0.0000


,fold,baseline_accuracy,override_accuracy,accuracy_change,predictions_changed,changed_accuracy_before,changed_accuracy_after
0,1,0.8380,0.8380,0.0000,0,NaN,NaN
1,2,0.8202,0.8202,0.0000,0,NaN,NaN
2,3,0.8202,0.8146,-0.0056,1,1.0,0.0
3,4,0.8315,0.8315,0.0000,0,NaN,NaN
4,5,0.8483,0.8483,0.0000,0,NaN,NaN
5,6,0.8547,0.8547,0.0000,0,NaN,NaN
6,7,0.8202,0.8202,0.0000,0,NaN,NaN
7,8,0.7978,0.7921,-0.0056,1,1.0,0.0
8,9,0.8202,0.8202,0.0000,0,NaN,NaN
9,10,0.8258,0.8258,0.0000,0,NaN,NaN


,fold,PassengerId,Name,PassengerType,FamilyGroup,known_family_outcome,known_label_count,original_prediction,override_prediction,actual,override_reason
0,3,183,"Asplund, Master. Clarence Gustaf Hugo",Boy,Asplund_7,1.0,3,0,1,0,Boy in all-survival WCG
1,8,183,"Asplund, Master. Clarence Gustaf Hugo",Boy,Asplund_7,1.0,3,0,1,0,Boy in all-survival WCG
2,14,183,"Asplund, Master. Clarence Gustaf Hugo",Boy,Asplund_7,1.0,3,0,1,0,Boy in all-survival WCG
3,16,183,"Asplund, Master. Clarence Gustaf Hugo",Boy,Asplund_7,1.0,2,0,1,0,Boy in all-survival WCG
4,21,183,"Asplund, Master. Clarence Gustaf Hugo",Boy,Asplund_7,1.0,2,0,1,0,Boy in all-survival WCG


In [14]:
print(
    "Unique passengers changed:",
    changed_rows_exp012_df["PassengerId"].nunique(),
)

print(
    "Unique family groups changed:",
    changed_rows_exp012_df["FamilyGroup"].nunique(),
)

display(
    changed_rows_exp012_df[
        [
            "PassengerId",
            "Name",
            "FamilyGroup",
            "known_label_count",
            "original_prediction",
            "override_prediction",
            "actual",
        ]
    ].drop_duplicates()
)

Unique passengers changed: 1
Unique family groups changed: 1


,PassengerId,Name,FamilyGroup,known_label_count,original_prediction,override_prediction,actual
0,183,"Asplund, Master. Clarence Gustaf Hugo",Asplund_7,3,0,1,0
3,183,"Asplund, Master. Clarence Gustaf Hugo",Asplund_7,2,0,1,0


In [11]:
unique_changes_exp012 = (
    changed_rows_exp012_df
    .groupby(
        [
            "PassengerId",
            "Name",
            "PassengerType",
            "FamilyGroup",
            "actual",
        ],
        as_index=False,
    )
    .agg(
        times_changed=("fold", "count"),
        average_known_labels=("known_label_count", "mean"),
        original_accuracy=(
            "original_prediction",
            lambda predictions: (
                predictions
                == changed_rows_exp012_df.loc[
                    predictions.index,
                    "actual",
                ]
            ).mean()
        ),
        override_accuracy=(
            "override_prediction",
            lambda predictions: (
                predictions
                == changed_rows_exp012_df.loc[
                    predictions.index,
                    "actual",
                ]
            ).mean()
        ),
    )
    .sort_values(
        ["times_changed", "FamilyGroup"],
        ascending=[False, True],
    )
)

print(
    "Repeated prediction changes:",
    len(changed_rows_exp012_df),
)

print(
    "Unique passengers changed:",
    unique_changes_exp012["PassengerId"].nunique(),
)

print(
    "Unique family groups involved:",
    unique_changes_exp012["FamilyGroup"].nunique(),
)

display(unique_changes_exp012)

Repeated prediction changes: 63
Unique passengers changed: 19
Unique family groups involved: 12


,PassengerId,Name,PassengerType,FamilyGroup,actual,times_changed,average_known_labels,original_accuracy,override_accuracy
6,183,"Asplund, Master. Clarence Gustaf Hugo",Boy,Asplund_7,0,5,2.6,1.0,0.0
9,363,"Barbara, Mrs. (Catherine David)",Woman,Barbara_2,0,5,1.0,0.0,1.0
14,703,"Barbara, Miss. Saiide",Woman,Barbara_2,0,5,1.0,0.0,1.0
1,112,"Zabour, Miss. Hileni",Woman,Zabour_2,0,5,1.0,0.0,1.0
7,241,"Zabour, Miss. Thamine",Woman,Zabour_2,0,5,1.0,0.0,1.0
4,141,"Boulos, Mrs. Joseph (Sultana)",Woman,Boulos_3,0,4,1.0,0.0,1.0
17,853,"Boulos, Miss. Nourelain",Woman,Boulos_3,0,4,1.0,0.0,1.0
2,114,"Jussila, Miss. Katriina",Woman,Jussila_2,0,4,1.0,0.0,1.0
10,403,"Jussila, Miss. Mari Aina",Woman,Jussila_2,0,4,1.0,0.0,1.0
3,126,"Nicola-Yarred, Master. Elias",Boy,Nicola-Yarred_2,1,4,1.0,0.0,1.0


In [16]:
# ============================================================
# EXP-012: CREATE KAGGLE TEST PREDICTIONS
# ============================================================

# Train the unchanged champion model on all labelled data.
final_model_exp012 = build_exp011_pipeline(full_features)

final_model_exp012.fit(
    train_features[full_features],
    y,
)

baseline_test_predictions_exp012 = final_model_exp012.predict(
    test_features[full_features]
)

override_test_predictions_exp012 = (
    baseline_test_predictions_exp012.copy()
)


# ------------------------------------------------------------
# Prepare relational information for train and test
# ------------------------------------------------------------

def prepare_relational_data(df):
    result = df.copy()

    result["Surname"] = (
        result["Name"]
        .str.split(",")
        .str[0]
        .str.strip()
    )

    result["CalculatedFamilySize"] = (
        result["SibSp"]
        + result["Parch"]
        + 1
    )

    result["FamilyGroup"] = (
        result["Surname"]
        + "_"
        + result["CalculatedFamilySize"].astype(str)
    )

    result["ExtractedTitle"] = (
        result["Name"]
        .str.extract(
            r",\s*([^.]*)\.",
            expand=False,
        )
        .str.strip()
    )

    result["PassengerType"] = np.select(
        [
            result["Sex"].eq("female"),
            result["ExtractedTitle"].eq("Master"),
        ],
        [
            "Woman",
            "Boy",
        ],
        default="Man",
    )

    return result


relation_train_full_exp012 = prepare_relational_data(train)
relation_test_exp012 = prepare_relational_data(test)

relation_train_full_exp012["Survived"] = y.to_numpy()


# ------------------------------------------------------------
# Learn consistent woman-child outcomes from full training data
# ------------------------------------------------------------

training_women_children_exp012 = (
    relation_train_full_exp012[
        relation_train_full_exp012[
            "PassengerType"
        ].isin(["Woman", "Boy"])
    ]
    .copy()
)

family_outcomes_full_exp012 = (
    training_women_children_exp012
    .groupby("FamilyGroup")["Survived"]
    .agg(["mean", "count", "nunique"])
)

reliable_family_outcomes_full_exp012 = (
    family_outcomes_full_exp012[
        family_outcomes_full_exp012["nunique"] == 1
    ]
)

family_survival_map_full_exp012 = (
    reliable_family_outcomes_full_exp012["mean"]
    .to_dict()
)

family_count_map_full_exp012 = (
    reliable_family_outcomes_full_exp012["count"]
    .to_dict()
)


# ------------------------------------------------------------
# Apply the validated minimum-one-relative rule
# ------------------------------------------------------------

test_changes_exp012 = []

for position, (_, passenger) in enumerate(
    relation_test_exp012.iterrows()
):
    family_group = passenger["FamilyGroup"]
    passenger_type = passenger["PassengerType"]

    if family_group not in family_survival_map_full_exp012:
        continue

    family_outcome = family_survival_map_full_exp012[
        family_group
    ]

    original_prediction = int(
        baseline_test_predictions_exp012[position]
    )

    new_prediction = original_prediction
    reason = None

    if (
        passenger_type == "Woman"
        and family_outcome == 0
    ):
        new_prediction = 0
        reason = "Woman in all-death WCG"

    elif (
        passenger_type == "Boy"
        and family_outcome == 1
    ):
        new_prediction = 1
        reason = "Boy in all-survival WCG"

    if new_prediction != original_prediction:
        override_test_predictions_exp012[position] = (
            new_prediction
        )

        test_changes_exp012.append(
            {
                "PassengerId": passenger["PassengerId"],
                "Name": passenger["Name"],
                "PassengerType": passenger["PassengerType"],
                "FamilyGroup": family_group,
                "known_family_outcome": family_outcome,
                "known_label_count": (
                    family_count_map_full_exp012[
                        family_group
                    ]
                ),
                "original_prediction": original_prediction,
                "override_prediction": new_prediction,
                "override_reason": reason,
            }
        )


test_changes_exp012_df = pd.DataFrame(
    test_changes_exp012
)

print(
    "Test predictions changed:",
    len(test_changes_exp012_df),
)

display(test_changes_exp012_df)

Test predictions changed: 4


,PassengerId,Name,PassengerType,FamilyGroup,known_family_outcome,known_label_count,original_prediction,override_prediction,override_reason
0,925,"Johnston, Mrs. Andrew G (Elizabeth Lily"" Watson)""",Woman,Johnston_4,0.0,1,1,0,Woman in all-death WCG
1,929,"Cacic, Miss. Manda",Woman,Cacic_1,0.0,1,1,0,Woman in all-death WCG
2,1172,"Oreskovic, Miss. Jelka",Woman,Oreskovic_1,0.0,1,1,0,Woman in all-death WCG
3,1176,"Rosblom, Miss. Salli Helena",Woman,Rosblom_3,0.0,1,1,0,Woman in all-death WCG


In [17]:
from pathlib import Path

# Create the submissions folder if it does not already exist.
submission_dir = Path("submissions")
submission_dir.mkdir(exist_ok=True)

# Build the Kaggle submission.
submission_exp012 = pd.DataFrame(
    {
        "PassengerId": test["PassengerId"],
        "Survived": override_test_predictions_exp012.astype(int),
    }
)

# Basic safety checks.
assert submission_exp012.shape == (418, 2)
assert submission_exp012["PassengerId"].is_unique
assert submission_exp012["Survived"].isin([0, 1]).all()
assert submission_exp012["Survived"].isna().sum() == 0

# Save the file.
submission_path_exp012 = (
    submission_dir
    / "exp012_random_forest_family_override.csv"
)

submission_exp012.to_csv(
    submission_path_exp012,
    index=False,
)

print("Submission created:", submission_path_exp012)
print("\nPrediction counts:")
print(submission_exp012["Survived"].value_counts().sort_index())

print("\nSubmission preview:")
display(submission_exp012.head(10))

Submission created: submissions\exp012_random_forest_family_override.csv

Prediction counts:
Survived
0    263
1    155
Name: count, dtype: int64

Submission preview:


,PassengerId,Survived
0,892,0
1,893,0
2,894,0
3,895,0
4,896,1
5,897,0
6,898,1
7,899,0
8,900,1
9,901,0


### Experiment Log

In [5]:
from pathlib import Path

import numpy as np
import pandas as pd


# ============================================================
# EXPERIMENT LOGGING FUNCTION
# ============================================================

experiment_path = Path("experiments.csv")


def log_experiment(
    experiment_id,
    description,
    model,
    features,
    validation_method,
    cv_scores,
    kaggle_score,
    changes,
    notes="",
    submission_file="",
):
    """
    Add or update one experiment in experiments.csv.

    Re-running an experiment with the same experiment_id replaces
    the previous record instead of creating a duplicate.
    """

    cv_scores = np.array(cv_scores, dtype=float)

    new_experiment = pd.DataFrame(
        [
            {
                "experiment_id": experiment_id,
                "description": description,
                "model": model,
                "features": features,
                "validation_method": validation_method,
                "cv_scores": ", ".join(
                    f"{score:.4f}" for score in cv_scores
                ),
                "cv_accuracy": cv_scores.mean(),
                "cv_std": cv_scores.std(),
                "kaggle_score": kaggle_score,
                "cv_kaggle_gap": cv_scores.mean() - kaggle_score,
                "changes": changes,
                "submission_file": submission_file,
                "notes": notes,
            }
        ]
    )

    if experiment_path.exists():
        experiments = pd.read_csv(experiment_path)

        # Remove an existing version of this experiment.
        experiments = experiments[
            experiments["experiment_id"] != experiment_id
        ]

        experiments = pd.concat(
            [experiments, new_experiment],
            ignore_index=True,
        )
    else:
        experiments = new_experiment

    experiments = experiments.sort_values(
        "experiment_id"
    ).reset_index(drop=True)

    experiments.to_csv(experiment_path, index=False)

    display(
        experiments.style.format(
            {
                "cv_accuracy": "{:.4f}",
                "cv_std": "{:.4f}",
                "kaggle_score": "{:.5f}",
                "cv_kaggle_gap": "{:.4f}",
            }
        )
    )

    return experiments

In [18]:
FEATURE_LIST = (
    "Pclass, Sex, Age, SibSp, Parch, Fare, Embarked, "
    "Title, FamilySize, IsAlone, Deck, FarePerPerson"
)


# ============================================================
# EXP-001: XGBOOST
# ============================================================

experiments = log_experiment(
    experiment_id="EXP-001",
    description="XGBoost baseline with engineered features",
    model="XGBClassifier",
    features=FEATURE_LIST,
    validation_method="5-fold StratifiedKFold",
    cv_scores=[
        0.8659,
        0.8483,
        0.8034,
        0.8258,
        0.8427,
    ],
    kaggle_score=0.76076,
    changes="Initial engineered-feature baseline",
    submission_file="submission_xgb_baseline.csv",
    notes=(
        "Highest local CV so far, but large CV-to-Kaggle gap. "
        "May be fitting training-specific nonlinear patterns."
    ),
)


# ============================================================
# EXP-002: LOGISTIC REGRESSION
# ============================================================

experiments = log_experiment(
    experiment_id="EXP-002",
    description="Logistic regression using identical engineered features",
    model="LogisticRegression",
    features=FEATURE_LIST,
    validation_method="5-fold StratifiedKFold",
    cv_scores=[
        0.8436,
        0.8258,
        0.7978,
        0.8315,
        0.8427,
    ],
    kaggle_score=0.76555,
    changes=(
        "Replaced XGBClassifier with LogisticRegression; "
        "all other modeling steps held constant"
    ),
    submission_file="submission_logistic.csv",
    notes=(
        "Lower CV than XGBoost but better Kaggle score and "
        "smaller validation-to-leaderboard gap. Current leader."
    ),
)


# ============================================================
# EXP-003: RANDOM FOREST
# ============================================================

experiments = log_experiment(
    experiment_id="EXP-003",
    description="Random forest using identical engineered features",
    model="RandomForestClassifier",
    features=FEATURE_LIST,
    validation_method="5-fold StratifiedKFold",
    cv_scores=[
        0.8380,
        0.8202,
        0.8202,
        0.8315,
        0.8483,
    ],
    kaggle_score=0.78708,
    changes=(
        "Replaced logistic regression with a regularized random forest; "
        "features, preprocessing, and validation folds held constant"
    ),
    submission_file="submission_random_forest.csv",
    notes=(
        "Best Kaggle score so far. Random forest differed from logistic "
        "regression on only 15 of 418 passengers. In 12 of the 15 cases, "
        "logistic predicted survival while random forest predicted death; "
        "10 of those passengers were male and 7 were first-class males. "
        "This suggests random forest captured interactions between sex, "
        "class, title, fare, and deck that additive logistic regression "
        "could not represent. Models agreed on 96.4% of test predictions."
    ),
)


# ============================================================
# EXP-004: LOGISTIC INTERACTIONS
# ============================================================

experiments = log_experiment(
    experiment_id="EXP-004",
    description=(
        "Logistic regression with targeted categorical "
        "interaction features"
    ),
    model="LogisticRegression",
    features=(
        FEATURE_LIST
        + ", Sex_Pclass, Sex_Title, Title_Pclass"
    ),
    validation_method="5-fold StratifiedKFold",
    cv_scores=[
        0.8268,
        0.8539,
        0.7978,
        0.8427,
        0.8483,
    ],
    kaggle_score=0.76315,
    changes=(
        "Added Sex_Pclass, Sex_Title, and Title_Pclass interaction "
        "features to baseline logistic regression; model settings, "
        "preprocessing approach, and validation folds held constant"
    ),
    submission_file="submission_logistic_interactions.csv",
    notes=(
        "Interaction features raised mean CV accuracy from 0.8283 to "
        "0.8339 but reduced Kaggle accuracy from 0.76555 to 0.76315. "
        "The CV standard deviation and CV-to-Kaggle gap also increased. "
        "The targeted interactions therefore fit the training folds "
        "better without transferring to the hidden test passengers. "
        "Random Forest remains the best model at 0.78708."
    ),
)


# ============================================================
# EXP-005: RF age imputation
# ============================================================

experiments = log_experiment(
    experiment_id="EXP-005",
    description=(
        "Random forest with title-and-class-based age imputation"
    ),
    model="RandomForestClassifier",
    features=FEATURE_LIST,
    validation_method="5-fold StratifiedKFold",
    cv_scores=[
        0.8380,
        0.8146,
        0.8258,
        0.8315,
        0.8371,
    ],
    kaggle_score=0.78229,
    changes=(
        "Replaced overall median age imputation with fold-safe grouped "
        "median imputation using Title and Pclass; random forest settings, "
        "features, and validation folds held constant"
    ),
    submission_file="submission_rf_grouped_age.csv",
    notes=(
        "Grouped age imputation reduced mean CV accuracy from 0.8316 "
        "to 0.8294 and Kaggle accuracy from 0.78708 to 0.78229. "
        "It changed only 4 of 418 test predictions versus EXP-003. "
        "Although CV variability decreased slightly, the changed "
        "predictions did not improve hidden-test performance. "
        "Reject EXP-005; EXP-003 remains champion."
    ),
)


# ============================================================
# EXP-006
# ============================================================

experiments = log_experiment(
    experiment_id="EXP-006",
    description=(
        "Compared ordinary stratified validation with "
        "family-group-aware validation"
    ),
    model="RandomForestClassifier",
    features=FEATURE_LIST,
    validation_method=(
        "5-fold StratifiedGroupKFold using surname and family size"
    ),
    cv_scores=[
        0.7486,
        0.8588,
        0.8276,
        0.8245,
        0.8757,
    ],
    kaggle_score=np.nan,
    changes=(
        "Held the EXP-003 Random Forest, features, and preprocessing "
        "constant; replaced StratifiedKFold with StratifiedGroupKFold. "
        "Likely family members were assigned to the same fold using "
        "surname and family size, while solo passengers received "
        "individual groups."
    ),
    submission_file="",
    notes=(
        "No Kaggle submission was needed because the final trained model "
        "and test predictions did not change. Group-aware mean accuracy "
        "was 0.8270 versus 0.8316 for ordinary validation, a reduction "
        "of only 0.0046. This does not support family leakage as the main "
        "cause of the CV-to-Kaggle gap. However, grouped validation was "
        "much less stable, with SD 0.0437 versus 0.0108. Only 12 of 891 "
        "out-of-fold predictions differed. Retain StratifiedKFold as the "
        "primary comparison method and use grouped validation as a "
        "secondary robustness check."
    ),
)


# ============================================================
# EXP-007
# ============================================================

experiments = log_experiment(
    experiment_id="EXP-007",
    description=(
        "Conservative Random Forest hyperparameter tuning with "
        "repeated CV and group-aware robustness validation"
    ),
    model="RandomForestClassifier",
    features=FEATURE_LIST,
    validation_method=(
        "GridSearchCV with 5-fold StratifiedKFold, followed by "
        "5x5 RepeatedStratifiedKFold and 5-fold "
        "StratifiedGroupKFold robustness validation"
    ),
    cv_scores=[
        0.7371,
        0.8588,
        0.8218,
        0.8298,
        0.8531,
    ],
    kaggle_score=np.nan,
    changes=(
        "Searched max_depth, min_samples_leaf, and max_features while "
        "holding the feature set, preprocessing, number of trees, "
        "min_samples_split, metric, and random seed constant. "
        "The selected candidate used max_depth=5, max_features=0.75, "
        "min_samples_leaf=6, and min_samples_split=8."
    ),
    submission_file="",
    notes=(
        "The tuned candidate improved repeated stratified CV by 0.0079 "
        "and had slightly better stability, but grouped CV declined from "
        "0.8270 to 0.8201, a loss of 0.0069. This exceeded the predefined "
        "maximum acceptable grouped-CV decline of 0.005, so no Kaggle "
        "submission was created. Reject EXP-007; EXP-003 remains champion."
    ),
)


# ============================================================
# EXP-008
# ============================================================

experiments = log_experiment(
    experiment_id="EXP-008",
    description=(
        "Random forest with fold-safe ticket-group and "
        "ticket-prefix features"
    ),
    model="RandomForestClassifier",
    features=(
        FEATURE_LIST
        + ", TicketGroupSize, SharedTicket, TicketPrefix"
    ),
    validation_method=(
        "5x5 RepeatedStratifiedKFold with secondary "
        "5-fold StratifiedGroupKFold robustness validation"
    ),
    cv_scores=[
        0.7314,
        0.8531,
        0.8276,
        0.8245,
        0.8701,
    ],
    kaggle_score=np.nan,
    changes=(
        "Added fold-safe TicketGroupSize, SharedTicket, and "
        "TicketPrefix features while holding the EXP-003 Random "
        "Forest parameters, original features, preprocessing, "
        "metric, and random seed constant."
    ),
    submission_file="",
    notes=(
        "Ticket features reduced repeated CV mean from 0.8271 to "
        "0.8224, a decline of 0.0047. Group-aware CV also declined "
        "from 0.8270 to 0.8213, a loss of 0.0057. Stability remained "
        "nearly unchanged. Only 5 of 418 test predictions differed "
        "from EXP-003. Both predefined performance checks failed, "
        "so no Kaggle submission was made. Reject EXP-008; EXP-003 "
        "remains champion."
    ),
)


# ============================================================
# EXP-009
# ============================================================

experiments = log_experiment(
    experiment_id="EXP-009",
    description=(
        "Random forest with fold-safe ticket-group size features "
        "but without ticket prefix"
    ),
    model="RandomForestClassifier",
    features=(
        FEATURE_LIST
        + ", TicketGroupSize, SharedTicket"
    ),
    validation_method=(
        "5x5 RepeatedStratifiedKFold with secondary "
        "5-fold StratifiedGroupKFold robustness validation"
    ),
    cv_scores=[
        0.7371,
        0.8475,
        0.8276,
        0.8351,
        0.8814,
    ],
    kaggle_score=np.nan,
    changes=(
        "Added fold-safe TicketGroupSize and SharedTicket features "
        "to the EXP-003 Random Forest while excluding TicketPrefix. "
        "Model parameters, original features, preprocessing, metric, "
        "and random seed were held constant."
    ),
    submission_file="",
    notes=(
        "Ticket-group features improved repeated CV mean from 0.8271 "
        "to 0.8301, a gain of 0.0029. This suggests TicketGroupSize "
        "and SharedTicket contain mild predictive signal and that "
        "TicketPrefix was the harmful component of EXP-008. However, "
        "group-aware CV declined slightly from 0.8270 to 0.8257, a "
        "loss of 0.0013, so the predefined robustness requirement "
        "was not met. Only 3 of 418 test predictions changed, all "
        "from death to survival. No Kaggle submission was made. "
        "Reject EXP-009; EXP-003 remains champion."
    ),
)


# ============================================================
# EXP-010
# ============================================================

experiments = log_experiment(
    experiment_id="EXP-010",
    description=(
        "Probability ensemble combining Random Forest and "
        "logistic regression"
    ),
    model=(
        "RandomForestClassifier + LogisticRegression"
    ),
    features=FEATURE_LIST,
    validation_method=(
        "5x5 RepeatedStratifiedKFold with secondary "
        "5-fold StratifiedGroupKFold robustness validation"
    ),
    cv_scores=best_ensemble_exp010["scores"],
    kaggle_score=0.77751,
    changes=(
        "Blended Random Forest and logistic regression survival "
        "probabilities using weights selected from a predetermined "
        "grid. The selected blend used 60% Random Forest and "
        "40% logistic regression, with the feature set, individual "
        "model parameters, preprocessing, threshold, metric, and "
        "random seed held constant."
    ),
    submission_file=(
        "submission_rf_logistic_ensemble_exp010.csv"
    ),
    notes=(
        "The ensemble passed all predefined local criteria. Repeated "
        "CV improved from 0.8271 to 0.8330, a gain of 0.0058, while "
        "group-aware CV improved from 0.8270 to 0.8348, a gain of "
        "0.0078. Repeated-CV variability also declined slightly. "
        "However, Kaggle accuracy was 0.77751, below EXP-003's "
        "0.78708 by 0.00957. The ensemble changed only 6 of 418 test "
        "predictions, and those changes were collectively harmful on "
        "the leaderboard. Reject EXP-010 as champion; retain EXP-003."
    ),
)


# ============================================================
# EXP-011: FEATURE ABLATION OF CHAMPION MODEL
# ============================================================

experiments = log_experiment(
    experiment_id="EXP-011",
    description=(
        "Engineered-feature ablation using the EXP-003 Random Forest"
    ),
    model="RandomForestClassifier",
    features=(
        "Pclass, Sex, Age, SibSp, Parch, Fare, Embarked, "
        "Title, FamilySize, IsAlone, Deck"
    ),
    validation_method=(
        "5x5 RepeatedStratifiedKFold with secondary "
        "5-fold StratifiedGroupKFold robustness validation"
    ),
    cv_scores=variant_scores_exp011["No FarePerPerson"],
    kaggle_score=np.nan,
    changes=(
        "Removed one engineered feature group at a time while holding "
        "the Random Forest parameters, preprocessing, folds, metric, "
        "and random seed constant."
    ),
    submission_file="",
    notes=(
        "Removing FarePerPerson produced the strongest ablation result. "
        "Repeated CV increased from 0.8271 to 0.8298, a gain of 0.0027, "
        "while group-aware CV remained unchanged at 0.8246. The full "
        "model beat the no-FarePerPerson version on only 20% of paired "
        "repeated-CV folds. Removing Title caused the largest decline, "
        "reducing repeated CV by 0.0150 and grouped CV by 0.0193. "
        "Removing FamilySize and IsAlone also weakened both validation "
        "methods. The no-FarePerPerson gain did not reach the predefined "
        "0.004 submission threshold, so no Kaggle submission was made. "
        "Retain EXP-003 as champion."
    ),
)



# ============================================================
# EXP-012: 
# ============================================================
experiments = log_experiment(
    experiment_id="EXP-012",
    description=(
        "Fold-safe woman-child family outcome overrides applied "
        "to EXP-003 Random Forest predictions"
    ),
    model=(
        "RandomForestClassifier with relational "
        "family-outcome post-processing"
    ),
    features=FEATURE_LIST,
    validation_method=(
        "5x5 RepeatedStratifiedKFold; family outcomes learned "
        "only from each fold's training rows"
    ),
    cv_scores=(
        fold_results_exp012_df[
            "override_accuracy"
        ].to_numpy()
    ),
    kaggle_score=0.79665,
    changes=(
        "Kept the EXP-003 Random Forest unchanged. After prediction, "
        "women were changed to death when known women or boys in the "
        "same surname-and-family-size group had all died. Boys were "
        "changed to survival when known women or boys in their group "
        "had all survived."
    ),
    submission_file=(
        "exp012_random_forest_family_override.csv"
    ),
    notes=(
        "Mean repeated-CV accuracy improved from 0.8271 to 0.8390, "
        "a gain of 0.0119. The override improved 21 of 25 folds, "
        "worsened none, and left four unchanged. Across 63 repeated-fold "
        "changes representing 19 unique passengers from 12 family groups, "
        "changed-row accuracy increased from 0.0794 before the override "
        "to 0.9206 after it. A stricter rule requiring two known relatives "
        "was rejected because it applied to only one unique passenger and "
        "reduced accuracy. The Kaggle score improved from EXP-003's "
        "0.78708 to 0.79665, a leaderboard gain of 0.00957. Promote "
        "EXP-012 to champion."
    ),
)

,experiment_id,description,model,features,validation_method,cv_accuracy,kaggle_score,notes,filename,cv_scores,cv_std,cv_kaggle_gap,changes,submission_file
0,EXP-001,XGBoost baseline with engineered features,XGBClassifier,"Pclass, Sex, Age, SibSp, Parch, Fare, Embarked, Title, FamilySize, IsAlone, Deck, FarePerPerson",5-fold StratifiedKFold,0.8372,0.76076,"Highest local CV so far, but large CV-to-Kaggle gap. May be fitting training-specific nonlinear patterns.",nan,"0.8659, 0.8483, 0.8034, 0.8258, 0.8427",0.0212,0.0765,Initial engineered-feature baseline,submission_xgb_baseline.csv
1,EXP-002,Logistic regression using identical engineered features,LogisticRegression,"Pclass, Sex, Age, SibSp, Parch, Fare, Embarked, Title, FamilySize, IsAlone, Deck, FarePerPerson",5-fold StratifiedKFold,0.8283,0.76555,Lower CV than XGBoost but better Kaggle score and smaller validation-to-leaderboard gap. Current leader.,nan,"0.8436, 0.8258, 0.7978, 0.8315, 0.8427",0.0167,0.0627,Replaced XGBClassifier with LogisticRegression; all other modeling steps held constant,submission_logistic.csv
2,EXP-003,Random forest using identical engineered features,RandomForestClassifier,"Pclass, Sex, Age, SibSp, Parch, Fare, Embarked, Title, FamilySize, IsAlone, Deck, FarePerPerson",5-fold StratifiedKFold,0.8316,0.78708,"Best Kaggle score so far. Random forest differed from logistic regression on only 15 of 418 passengers. In 12 of the 15 cases, logistic predicted survival while random forest predicted death; 10 of those passengers were male and 7 were first-class males. This suggests random forest captured interactions between sex, class, title, fare, and deck that additive logistic regression could not represent. Models agreed on 96.4% of test predictions.",nan,"0.8380, 0.8202, 0.8202, 0.8315, 0.8483",0.0108,0.0446,"Replaced logistic regression with a regularized random forest; features, preprocessing, and validation folds held constant",submission_random_forest.csv
3,EXP-004,Logistic regression with targeted categorical interaction features,LogisticRegression,"Pclass, Sex, Age, SibSp, Parch, Fare, Embarked, Title, FamilySize, IsAlone, Deck, FarePerPerson, Sex_Pclass, Sex_Title, Title_Pclass",5-fold StratifiedKFold,0.8339,0.76315,Interaction features raised mean CV accuracy from 0.8283 to 0.8339 but reduced Kaggle accuracy from 0.76555 to 0.76315. The CV standard deviation and CV-to-Kaggle gap also increased. The targeted interactions therefore fit the training folds better without transferring to the hidden test passengers. Random Forest remains the best model at 0.78708.,nan,"0.8268, 0.8539, 0.7978, 0.8427, 0.8483",0.0202,0.0707,"Added Sex_Pclass, Sex_Title, and Title_Pclass interaction features to baseline logistic regression; model settings, preprocessing approach, and validation folds held constant",submission_logistic_interactions.csv
4,EXP-005,Random forest with title-and-class-based age imputation,RandomForestClassifier,"Pclass, Sex, Age, SibSp, Parch, Fare, Embarked, Title, FamilySize, IsAlone, Deck, FarePerPerson",5-fold StratifiedKFold,0.8294,0.78229,"Grouped age imputation reduced mean CV accuracy from 0.8316 to 0.8294 and Kaggle accuracy from 0.78708 to 0.78229. It changed only 4 of 418 test predictions versus EXP-003. Although CV variability decreased slightly, the changed predictions did not improve hidden-test performance. Reject EXP-005; EXP-003 remains champion.",nan,"0.8380, 0.8146, 0.8258, 0.8315, 0.8371",0.0086,0.0471,"Replaced overall median age imputation with fold-safe grouped median imputation using Title and Pclass; random forest settings, features, and validation folds held constant",submission_rf_grouped_age.csv
5,EXP-006,Compared ordinary stratified validation with family-group-aware validation,RandomForestClassifier,"Pclass, Sex, Age, SibSp, Parch, Fare, Embarked, Title, FamilySize, IsAlone, Deck, FarePerPerson",5-fold StratifiedGroupKFold using surname and family size,0.8270,nan,"No Kaggle submission was needed because the final tra

,experiment_id,description,model,features,validation_method,cv_accuracy,kaggle_score,notes,filename,cv_scores,cv_std,cv_kaggle_gap,changes,submission_file
0,EXP-001,XGBoost baseline with engineered features,XGBClassifier,"Pclass, Sex, Age, SibSp, Parch, Fare, Embarked, Title, FamilySize, IsAlone, Deck, FarePerPerson",5-fold StratifiedKFold,0.8372,0.76076,"Highest local CV so far, but large CV-to-Kaggle gap. May be fitting training-specific nonlinear patterns.",nan,"0.8659, 0.8483, 0.8034, 0.8258, 0.8427",0.0212,0.0765,Initial engineered-feature baseline,submission_xgb_baseline.csv
1,EXP-002,Logistic regression using identical engineered features,LogisticRegression,"Pclass, Sex, Age, SibSp, Parch, Fare, Embarked, Title, FamilySize, IsAlone, Deck, FarePerPerson",5-fold StratifiedKFold,0.8283,0.76555,Lower CV than XGBoost but better Kaggle score and smaller validation-to-leaderboard gap. Current leader.,nan,"0.8436, 0.8258, 0.7978, 0.8315, 0.8427",0.0167,0.0627,Replaced XGBClassifier with LogisticRegression; all other modeling steps held constant,submission_logistic.csv
2,EXP-003,Random forest using identical engineered features,RandomForestClassifier,"Pclass, Sex, Age, SibSp, Parch, Fare, Embarked, Title, FamilySize, IsAlone, Deck, FarePerPerson",5-fold StratifiedKFold,0.8316,0.78708,"Best Kaggle score so far. Random forest differed from logistic regression on only 15 of 418 passengers. In 12 of the 15 cases, logistic predicted survival while random forest predicted death; 10 of those passengers were male and 7 were first-class males. This suggests random forest captured interactions between sex, class, title, fare, and deck that additive logistic regression could not represent. Models agreed on 96.4% of test predictions.",nan,"0.8380, 0.8202, 0.8202, 0.8315, 0.8483",0.0108,0.0446,"Replaced logistic regression with a regularized random forest; features, preprocessing, and validation folds held constant",submission_random_forest.csv
3,EXP-004,Logistic regression with targeted categorical interaction features,LogisticRegression,"Pclass, Sex, Age, SibSp, Parch, Fare, Embarked, Title, FamilySize, IsAlone, Deck, FarePerPerson, Sex_Pclass, Sex_Title, Title_Pclass",5-fold StratifiedKFold,0.8339,0.76315,Interaction features raised mean CV accuracy from 0.8283 to 0.8339 but reduced Kaggle accuracy from 0.76555 to 0.76315. The CV standard deviation and CV-to-Kaggle gap also increased. The targeted interactions therefore fit the training folds better without transferring to the hidden test passengers. Random Forest remains the best model at 0.78708.,nan,"0.8268, 0.8539, 0.7978, 0.8427, 0.8483",0.0202,0.0707,"Added Sex_Pclass, Sex_Title, and Title_Pclass interaction features to baseline logistic regression; model settings, preprocessing approach, and validation folds held constant",submission_logistic_interactions.csv
4,EXP-005,Random forest with title-and-class-based age imputation,RandomForestClassifier,"Pclass, Sex, Age, SibSp, Parch, Fare, Embarked, Title, FamilySize, IsAlone, Deck, FarePerPerson",5-fold StratifiedKFold,0.8294,0.78229,"Grouped age imputation reduced mean CV accuracy from 0.8316 to 0.8294 and Kaggle accuracy from 0.78708 to 0.78229. It changed only 4 of 418 test predictions versus EXP-003. Although CV variability decreased slightly, the changed predictions did not improve hidden-test performance. Reject EXP-005; EXP-003 remains champion.",nan,"0.8380, 0.8146, 0.8258, 0.8315, 0.8371",0.0086,0.0471,"Replaced overall median age imputation with fold-safe grouped median imputation using Title and Pclass; random forest settings, features, and validation folds held constant",submission_rf_grouped_age.csv
5,EXP-006,Compared ordinary stratified validation with family-group-aware validation,RandomForestClassifier,"Pclass, Sex, Age, SibSp, Parch, Fare, Embarked, Title, FamilySize, IsAlone, Deck, FarePerPerson",5-fold StratifiedGroupKFold using surname and family size,0.8270,nan,"No Kaggle submission was needed because the final tra

,experiment_id,description,model,features,validation_method,cv_accuracy,kaggle_score,notes,filename,cv_scores,cv_std,cv_kaggle_gap,changes,submission_file
0,EXP-001,XGBoost baseline with engineered features,XGBClassifier,"Pclass, Sex, Age, SibSp, Parch, Fare, Embarked, Title, FamilySize, IsAlone, Deck, FarePerPerson",5-fold StratifiedKFold,0.8372,0.76076,"Highest local CV so far, but large CV-to-Kaggle gap. May be fitting training-specific nonlinear patterns.",nan,"0.8659, 0.8483, 0.8034, 0.8258, 0.8427",0.0212,0.0765,Initial engineered-feature baseline,submission_xgb_baseline.csv
1,EXP-002,Logistic regression using identical engineered features,LogisticRegression,"Pclass, Sex, Age, SibSp, Parch, Fare, Embarked, Title, FamilySize, IsAlone, Deck, FarePerPerson",5-fold StratifiedKFold,0.8283,0.76555,Lower CV than XGBoost but better Kaggle score and smaller validation-to-leaderboard gap. Current leader.,nan,"0.8436, 0.8258, 0.7978, 0.8315, 0.8427",0.0167,0.0627,Replaced XGBClassifier with LogisticRegression; all other modeling steps held constant,submission_logistic.csv
2,EXP-003,Random forest using identical engineered features,RandomForestClassifier,"Pclass, Sex, Age, SibSp, Parch, Fare, Embarked, Title, FamilySize, IsAlone, Deck, FarePerPerson",5-fold StratifiedKFold,0.8316,0.78708,"Best Kaggle score so far. Random forest differed from logistic regression on only 15 of 418 passengers. In 12 of the 15 cases, logistic predicted survival while random forest predicted death; 10 of those passengers were male and 7 were first-class males. This suggests random forest captured interactions between sex, class, title, fare, and deck that additive logistic regression could not represent. Models agreed on 96.4% of test predictions.",nan,"0.8380, 0.8202, 0.8202, 0.8315, 0.8483",0.0108,0.0446,"Replaced logistic regression with a regularized random forest; features, preprocessing, and validation folds held constant",submission_random_forest.csv
3,EXP-004,Logistic regression with targeted categorical interaction features,LogisticRegression,"Pclass, Sex, Age, SibSp, Parch, Fare, Embarked, Title, FamilySize, IsAlone, Deck, FarePerPerson, Sex_Pclass, Sex_Title, Title_Pclass",5-fold StratifiedKFold,0.8339,0.76315,Interaction features raised mean CV accuracy from 0.8283 to 0.8339 but reduced Kaggle accuracy from 0.76555 to 0.76315. The CV standard deviation and CV-to-Kaggle gap also increased. The targeted interactions therefore fit the training folds better without transferring to the hidden test passengers. Random Forest remains the best model at 0.78708.,nan,"0.8268, 0.8539, 0.7978, 0.8427, 0.8483",0.0202,0.0707,"Added Sex_Pclass, Sex_Title, and Title_Pclass interaction features to baseline logistic regression; model settings, preprocessing approach, and validation folds held constant",submission_logistic_interactions.csv
4,EXP-005,Random forest with title-and-class-based age imputation,RandomForestClassifier,"Pclass, Sex, Age, SibSp, Parch, Fare, Embarked, Title, FamilySize, IsAlone, Deck, FarePerPerson",5-fold StratifiedKFold,0.8294,0.78229,"Grouped age imputation reduced mean CV accuracy from 0.8316 to 0.8294 and Kaggle accuracy from 0.78708 to 0.78229. It changed only 4 of 418 test predictions versus EXP-003. Although CV variability decreased slightly, the changed predictions did not improve hidden-test performance. Reject EXP-005; EXP-003 remains champion.",nan,"0.8380, 0.8146, 0.8258, 0.8315, 0.8371",0.0086,0.0471,"Replaced overall median age imputation with fold-safe grouped median imputation using Title and Pclass; random forest settings, features, and validation folds held constant",submission_rf_grouped_age.csv
5,EXP-006,Compared ordinary stratified validation with family-group-aware validation,RandomForestClassifier,"Pclass, Sex, Age, SibSp, Parch, Fare, Embarked, Title, FamilySize, IsAlone, Deck, FarePerPerson",5-fold StratifiedGroupKFold using surname and family size,0.8270,nan,"No Kaggle submission was needed because the final tra

,experiment_id,description,model,features,validation_method,cv_accuracy,kaggle_score,notes,filename,cv_scores,cv_std,cv_kaggle_gap,changes,submission_file
0,EXP-001,XGBoost baseline with engineered features,XGBClassifier,"Pclass, Sex, Age, SibSp, Parch, Fare, Embarked, Title, FamilySize, IsAlone, Deck, FarePerPerson",5-fold StratifiedKFold,0.8372,0.76076,"Highest local CV so far, but large CV-to-Kaggle gap. May be fitting training-specific nonlinear patterns.",nan,"0.8659, 0.8483, 0.8034, 0.8258, 0.8427",0.0212,0.0765,Initial engineered-feature baseline,submission_xgb_baseline.csv
1,EXP-002,Logistic regression using identical engineered features,LogisticRegression,"Pclass, Sex, Age, SibSp, Parch, Fare, Embarked, Title, FamilySize, IsAlone, Deck, FarePerPerson",5-fold StratifiedKFold,0.8283,0.76555,Lower CV than XGBoost but better Kaggle score and smaller validation-to-leaderboard gap. Current leader.,nan,"0.8436, 0.8258, 0.7978, 0.8315, 0.8427",0.0167,0.0627,Replaced XGBClassifier with LogisticRegression; all other modeling steps held constant,submission_logistic.csv
2,EXP-003,Random forest using identical engineered features,RandomForestClassifier,"Pclass, Sex, Age, SibSp, Parch, Fare, Embarked, Title, FamilySize, IsAlone, Deck, FarePerPerson",5-fold StratifiedKFold,0.8316,0.78708,"Best Kaggle score so far. Random forest differed from logistic regression on only 15 of 418 passengers. In 12 of the 15 cases, logistic predicted survival while random forest predicted death; 10 of those passengers were male and 7 were first-class males. This suggests random forest captured interactions between sex, class, title, fare, and deck that additive logistic regression could not represent. Models agreed on 96.4% of test predictions.",nan,"0.8380, 0.8202, 0.8202, 0.8315, 0.8483",0.0108,0.0446,"Replaced logistic regression with a regularized random forest; features, preprocessing, and validation folds held constant",submission_random_forest.csv
3,EXP-004,Logistic regression with targeted categorical interaction features,LogisticRegression,"Pclass, Sex, Age, SibSp, Parch, Fare, Embarked, Title, FamilySize, IsAlone, Deck, FarePerPerson, Sex_Pclass, Sex_Title, Title_Pclass",5-fold StratifiedKFold,0.8339,0.76315,Interaction features raised mean CV accuracy from 0.8283 to 0.8339 but reduced Kaggle accuracy from 0.76555 to 0.76315. The CV standard deviation and CV-to-Kaggle gap also increased. The targeted interactions therefore fit the training folds better without transferring to the hidden test passengers. Random Forest remains the best model at 0.78708.,nan,"0.8268, 0.8539, 0.7978, 0.8427, 0.8483",0.0202,0.0708,"Added Sex_Pclass, Sex_Title, and Title_Pclass interaction features to baseline logistic regression; model settings, preprocessing approach, and validation folds held constant",submission_logistic_interactions.csv
4,EXP-005,Random forest with title-and-class-based age imputation,RandomForestClassifier,"Pclass, Sex, Age, SibSp, Parch, Fare, Embarked, Title, FamilySize, IsAlone, Deck, FarePerPerson",5-fold StratifiedKFold,0.8294,0.78229,"Grouped age imputation reduced mean CV accuracy from 0.8316 to 0.8294 and Kaggle accuracy from 0.78708 to 0.78229. It changed only 4 of 418 test predictions versus EXP-003. Although CV variability decreased slightly, the changed predictions did not improve hidden-test performance. Reject EXP-005; EXP-003 remains champion.",nan,"0.8380, 0.8146, 0.8258, 0.8315, 0.8371",0.0086,0.0471,"Replaced overall median age imputation with fold-safe grouped median imputation using Title and Pclass; random forest settings, features, and validation folds held constant",submission_rf_grouped_age.csv
5,EXP-006,Compared ordinary stratified validation with family-group-aware validation,RandomForestClassifier,"Pclass, Sex, Age, SibSp, Parch, Fare, Embarked, Title, FamilySize, IsAlone, Deck, FarePerPerson",5-fold StratifiedGroupKFold using surname and family size,0.8270,nan,"No Kaggle submission was needed because the final tra

,experiment_id,description,model,features,validation_method,cv_accuracy,kaggle_score,notes,filename,cv_scores,cv_std,cv_kaggle_gap,changes,submission_file
0,EXP-001,XGBoost baseline with engineered features,XGBClassifier,"Pclass, Sex, Age, SibSp, Parch, Fare, Embarked, Title, FamilySize, IsAlone, Deck, FarePerPerson",5-fold StratifiedKFold,0.8372,0.76076,"Highest local CV so far, but large CV-to-Kaggle gap. May be fitting training-specific nonlinear patterns.",nan,"0.8659, 0.8483, 0.8034, 0.8258, 0.8427",0.0212,0.0765,Initial engineered-feature baseline,submission_xgb_baseline.csv
1,EXP-002,Logistic regression using identical engineered features,LogisticRegression,"Pclass, Sex, Age, SibSp, Parch, Fare, Embarked, Title, FamilySize, IsAlone, Deck, FarePerPerson",5-fold StratifiedKFold,0.8283,0.76555,Lower CV than XGBoost but better Kaggle score and smaller validation-to-leaderboard gap. Current leader.,nan,"0.8436, 0.8258, 0.7978, 0.8315, 0.8427",0.0167,0.0627,Replaced XGBClassifier with LogisticRegression; all other modeling steps held constant,submission_logistic.csv
2,EXP-003,Random forest using identical engineered features,RandomForestClassifier,"Pclass, Sex, Age, SibSp, Parch, Fare, Embarked, Title, FamilySize, IsAlone, Deck, FarePerPerson",5-fold StratifiedKFold,0.8316,0.78708,"Best Kaggle score so far. Random forest differed from logistic regression on only 15 of 418 passengers. In 12 of the 15 cases, logistic predicted survival while random forest predicted death; 10 of those passengers were male and 7 were first-class males. This suggests random forest captured interactions between sex, class, title, fare, and deck that additive logistic regression could not represent. Models agreed on 96.4% of test predictions.",nan,"0.8380, 0.8202, 0.8202, 0.8315, 0.8483",0.0108,0.0446,"Replaced logistic regression with a regularized random forest; features, preprocessing, and validation folds held constant",submission_random_forest.csv
3,EXP-004,Logistic regression with targeted categorical interaction features,LogisticRegression,"Pclass, Sex, Age, SibSp, Parch, Fare, Embarked, Title, FamilySize, IsAlone, Deck, FarePerPerson, Sex_Pclass, Sex_Title, Title_Pclass",5-fold StratifiedKFold,0.8339,0.76315,Interaction features raised mean CV accuracy from 0.8283 to 0.8339 but reduced Kaggle accuracy from 0.76555 to 0.76315. The CV standard deviation and CV-to-Kaggle gap also increased. The targeted interactions therefore fit the training folds better without transferring to the hidden test passengers. Random Forest remains the best model at 0.78708.,nan,"0.8268, 0.8539, 0.7978, 0.8427, 0.8483",0.0202,0.0707,"Added Sex_Pclass, Sex_Title, and Title_Pclass interaction features to baseline logistic regression; model settings, preprocessing approach, and validation folds held constant",submission_logistic_interactions.csv
4,EXP-005,Random forest with title-and-class-based age imputation,RandomForestClassifier,"Pclass, Sex, Age, SibSp, Parch, Fare, Embarked, Title, FamilySize, IsAlone, Deck, FarePerPerson",5-fold StratifiedKFold,0.8294,0.78229,"Grouped age imputation reduced mean CV accuracy from 0.8316 to 0.8294 and Kaggle accuracy from 0.78708 to 0.78229. It changed only 4 of 418 test predictions versus EXP-003. Although CV variability decreased slightly, the changed predictions did not improve hidden-test performance. Reject EXP-005; EXP-003 remains champion.",nan,"0.8380, 0.8146, 0.8258, 0.8315, 0.8371",0.0086,0.0471,"Replaced overall median age imputation with fold-safe grouped median imputation using Title and Pclass; random forest settings, features, and validation folds held constant",submission_rf_grouped_age.csv
5,EXP-006,Compared ordinary stratified validation with family-group-aware validation,RandomForestClassifier,"Pclass, Sex, Age, SibSp, Parch, Fare, Embarked, Title, FamilySize, IsAlone, Deck, FarePerPerson",5-fold StratifiedGroupKFold using surname and family size,0.8270,nan,"No Kaggle submission was needed because the final tra

,experiment_id,description,model,features,validation_method,cv_accuracy,kaggle_score,notes,filename,cv_scores,cv_std,cv_kaggle_gap,changes,submission_file
0,EXP-001,XGBoost baseline with engineered features,XGBClassifier,"Pclass, Sex, Age, SibSp, Parch, Fare, Embarked, Title, FamilySize, IsAlone, Deck, FarePerPerson",5-fold StratifiedKFold,0.8372,0.76076,"Highest local CV so far, but large CV-to-Kaggle gap. May be fitting training-specific nonlinear patterns.",nan,"0.8659, 0.8483, 0.8034, 0.8258, 0.8427",0.0212,0.0765,Initial engineered-feature baseline,submission_xgb_baseline.csv
1,EXP-002,Logistic regression using identical engineered features,LogisticRegression,"Pclass, Sex, Age, SibSp, Parch, Fare, Embarked, Title, FamilySize, IsAlone, Deck, FarePerPerson",5-fold StratifiedKFold,0.8283,0.76555,Lower CV than XGBoost but better Kaggle score and smaller validation-to-leaderboard gap. Current leader.,nan,"0.8436, 0.8258, 0.7978, 0.8315, 0.8427",0.0167,0.0627,Replaced XGBClassifier with LogisticRegression; all other modeling steps held constant,submission_logistic.csv
2,EXP-003,Random forest using identical engineered features,RandomForestClassifier,"Pclass, Sex, Age, SibSp, Parch, Fare, Embarked, Title, FamilySize, IsAlone, Deck, FarePerPerson",5-fold StratifiedKFold,0.8316,0.78708,"Best Kaggle score so far. Random forest differed from logistic regression on only 15 of 418 passengers. In 12 of the 15 cases, logistic predicted survival while random forest predicted death; 10 of those passengers were male and 7 were first-class males. This suggests random forest captured interactions between sex, class, title, fare, and deck that additive logistic regression could not represent. Models agreed on 96.4% of test predictions.",nan,"0.8380, 0.8202, 0.8202, 0.8315, 0.8483",0.0108,0.0446,"Replaced logistic regression with a regularized random forest; features, preprocessing, and validation folds held constant",submission_random_forest.csv
3,EXP-004,Logistic regression with targeted categorical interaction features,LogisticRegression,"Pclass, Sex, Age, SibSp, Parch, Fare, Embarked, Title, FamilySize, IsAlone, Deck, FarePerPerson, Sex_Pclass, Sex_Title, Title_Pclass",5-fold StratifiedKFold,0.8339,0.76315,Interaction features raised mean CV accuracy from 0.8283 to 0.8339 but reduced Kaggle accuracy from 0.76555 to 0.76315. The CV standard deviation and CV-to-Kaggle gap also increased. The targeted interactions therefore fit the training folds better without transferring to the hidden test passengers. Random Forest remains the best model at 0.78708.,nan,"0.8268, 0.8539, 0.7978, 0.8427, 0.8483",0.0202,0.0707,"Added Sex_Pclass, Sex_Title, and Title_Pclass interaction features to baseline logistic regression; model settings, preprocessing approach, and validation folds held constant",submission_logistic_interactions.csv
4,EXP-005,Random forest with title-and-class-based age imputation,RandomForestClassifier,"Pclass, Sex, Age, SibSp, Parch, Fare, Embarked, Title, FamilySize, IsAlone, Deck, FarePerPerson",5-fold StratifiedKFold,0.8294,0.78229,"Grouped age imputation reduced mean CV accuracy from 0.8316 to 0.8294 and Kaggle accuracy from 0.78708 to 0.78229. It changed only 4 of 418 test predictions versus EXP-003. Although CV variability decreased slightly, the changed predictions did not improve hidden-test performance. Reject EXP-005; EXP-003 remains champion.",nan,"0.8380, 0.8146, 0.8258, 0.8315, 0.8371",0.0086,0.0471,"Replaced overall median age imputation with fold-safe grouped median imputation using Title and Pclass; random forest settings, features, and validation folds held constant",submission_rf_grouped_age.csv
5,EXP-006,Compared ordinary stratified validation with family-group-aware validation,RandomForestClassifier,"Pclass, Sex, Age, SibSp, Parch, Fare, Embarked, Title, FamilySize, IsAlone, Deck, FarePerPerson",5-fold StratifiedGroupKFold using surname and family size,0.8270,nan,"No Kaggle submission was needed because the final tra

,experiment_id,description,model,features,validation_method,cv_accuracy,kaggle_score,notes,filename,cv_scores,cv_std,cv_kaggle_gap,changes,submission_file
0,EXP-001,XGBoost baseline with engineered features,XGBClassifier,"Pclass, Sex, Age, SibSp, Parch, Fare, Embarked, Title, FamilySize, IsAlone, Deck, FarePerPerson",5-fold StratifiedKFold,0.8372,0.76076,"Highest local CV so far, but large CV-to-Kaggle gap. May be fitting training-specific nonlinear patterns.",nan,"0.8659, 0.8483, 0.8034, 0.8258, 0.8427",0.0212,0.0765,Initial engineered-feature baseline,submission_xgb_baseline.csv
1,EXP-002,Logistic regression using identical engineered features,LogisticRegression,"Pclass, Sex, Age, SibSp, Parch, Fare, Embarked, Title, FamilySize, IsAlone, Deck, FarePerPerson",5-fold StratifiedKFold,0.8283,0.76555,Lower CV than XGBoost but better Kaggle score and smaller validation-to-leaderboard gap. Current leader.,nan,"0.8436, 0.8258, 0.7978, 0.8315, 0.8427",0.0167,0.0627,Replaced XGBClassifier with LogisticRegression; all other modeling steps held constant,submission_logistic.csv
2,EXP-003,Random forest using identical engineered features,RandomForestClassifier,"Pclass, Sex, Age, SibSp, Parch, Fare, Embarked, Title, FamilySize, IsAlone, Deck, FarePerPerson",5-fold StratifiedKFold,0.8316,0.78708,"Best Kaggle score so far. Random forest differed from logistic regression on only 15 of 418 passengers. In 12 of the 15 cases, logistic predicted survival while random forest predicted death; 10 of those passengers were male and 7 were first-class males. This suggests random forest captured interactions between sex, class, title, fare, and deck that additive logistic regression could not represent. Models agreed on 96.4% of test predictions.",nan,"0.8380, 0.8202, 0.8202, 0.8315, 0.8483",0.0108,0.0446,"Replaced logistic regression with a regularized random forest; features, preprocessing, and validation folds held constant",submission_random_forest.csv
3,EXP-004,Logistic regression with targeted categorical interaction features,LogisticRegression,"Pclass, Sex, Age, SibSp, Parch, Fare, Embarked, Title, FamilySize, IsAlone, Deck, FarePerPerson, Sex_Pclass, Sex_Title, Title_Pclass",5-fold StratifiedKFold,0.8339,0.76315,Interaction features raised mean CV accuracy from 0.8283 to 0.8339 but reduced Kaggle accuracy from 0.76555 to 0.76315. The CV standard deviation and CV-to-Kaggle gap also increased. The targeted interactions therefore fit the training folds better without transferring to the hidden test passengers. Random Forest remains the best model at 0.78708.,nan,"0.8268, 0.8539, 0.7978, 0.8427, 0.8483",0.0202,0.0707,"Added Sex_Pclass, Sex_Title, and Title_Pclass interaction features to baseline logistic regression; model settings, preprocessing approach, and validation folds held constant",submission_logistic_interactions.csv
4,EXP-005,Random forest with title-and-class-based age imputation,RandomForestClassifier,"Pclass, Sex, Age, SibSp, Parch, Fare, Embarked, Title, FamilySize, IsAlone, Deck, FarePerPerson",5-fold StratifiedKFold,0.8294,0.78229,"Grouped age imputation reduced mean CV accuracy from 0.8316 to 0.8294 and Kaggle accuracy from 0.78708 to 0.78229. It changed only 4 of 418 test predictions versus EXP-003. Although CV variability decreased slightly, the changed predictions did not improve hidden-test performance. Reject EXP-005; EXP-003 remains champion.",nan,"0.8380, 0.8146, 0.8258, 0.8315, 0.8371",0.0086,0.0471,"Replaced overall median age imputation with fold-safe grouped median imputation using Title and Pclass; random forest settings, features, and validation folds held constant",submission_rf_grouped_age.csv
5,EXP-006,Compared ordinary stratified validation with family-group-aware validation,RandomForestClassifier,"Pclass, Sex, Age, SibSp, Parch, Fare, Embarked, Title, FamilySize, IsAlone, Deck, FarePerPerson",5-fold StratifiedGroupKFold using surname and family size,0.8270,nan,"No Kaggle submission was needed because the final tra

,experiment_id,description,model,features,validation_method,cv_accuracy,kaggle_score,notes,filename,cv_scores,cv_std,cv_kaggle_gap,changes,submission_file
0,EXP-001,XGBoost baseline with engineered features,XGBClassifier,"Pclass, Sex, Age, SibSp, Parch, Fare, Embarked, Title, FamilySize, IsAlone, Deck, FarePerPerson",5-fold StratifiedKFold,0.8372,0.76076,"Highest local CV so far, but large CV-to-Kaggle gap. May be fitting training-specific nonlinear patterns.",nan,"0.8659, 0.8483, 0.8034, 0.8258, 0.8427",0.0212,0.0765,Initial engineered-feature baseline,submission_xgb_baseline.csv
1,EXP-002,Logistic regression using identical engineered features,LogisticRegression,"Pclass, Sex, Age, SibSp, Parch, Fare, Embarked, Title, FamilySize, IsAlone, Deck, FarePerPerson",5-fold StratifiedKFold,0.8283,0.76555,Lower CV than XGBoost but better Kaggle score and smaller validation-to-leaderboard gap. Current leader.,nan,"0.8436, 0.8258, 0.7978, 0.8315, 0.8427",0.0167,0.0627,Replaced XGBClassifier with LogisticRegression; all other modeling steps held constant,submission_logistic.csv
2,EXP-003,Random forest using identical engineered features,RandomForestClassifier,"Pclass, Sex, Age, SibSp, Parch, Fare, Embarked, Title, FamilySize, IsAlone, Deck, FarePerPerson",5-fold StratifiedKFold,0.8316,0.78708,"Best Kaggle score so far. Random forest differed from logistic regression on only 15 of 418 passengers. In 12 of the 15 cases, logistic predicted survival while random forest predicted death; 10 of those passengers were male and 7 were first-class males. This suggests random forest captured interactions between sex, class, title, fare, and deck that additive logistic regression could not represent. Models agreed on 96.4% of test predictions.",nan,"0.8380, 0.8202, 0.8202, 0.8315, 0.8483",0.0108,0.0446,"Replaced logistic regression with a regularized random forest; features, preprocessing, and validation folds held constant",submission_random_forest.csv
3,EXP-004,Logistic regression with targeted categorical interaction features,LogisticRegression,"Pclass, Sex, Age, SibSp, Parch, Fare, Embarked, Title, FamilySize, IsAlone, Deck, FarePerPerson, Sex_Pclass, Sex_Title, Title_Pclass",5-fold StratifiedKFold,0.8339,0.76315,Interaction features raised mean CV accuracy from 0.8283 to 0.8339 but reduced Kaggle accuracy from 0.76555 to 0.76315. The CV standard deviation and CV-to-Kaggle gap also increased. The targeted interactions therefore fit the training folds better without transferring to the hidden test passengers. Random Forest remains the best model at 0.78708.,nan,"0.8268, 0.8539, 0.7978, 0.8427, 0.8483",0.0202,0.0707,"Added Sex_Pclass, Sex_Title, and Title_Pclass interaction features to baseline logistic regression; model settings, preprocessing approach, and validation folds held constant",submission_logistic_interactions.csv
4,EXP-005,Random forest with title-and-class-based age imputation,RandomForestClassifier,"Pclass, Sex, Age, SibSp, Parch, Fare, Embarked, Title, FamilySize, IsAlone, Deck, FarePerPerson",5-fold StratifiedKFold,0.8294,0.78229,"Grouped age imputation reduced mean CV accuracy from 0.8316 to 0.8294 and Kaggle accuracy from 0.78708 to 0.78229. It changed only 4 of 418 test predictions versus EXP-003. Although CV variability decreased slightly, the changed predictions did not improve hidden-test performance. Reject EXP-005; EXP-003 remains champion.",nan,"0.8380, 0.8146, 0.8258, 0.8315, 0.8371",0.0086,0.0471,"Replaced overall median age imputation with fold-safe grouped median imputation using Title and Pclass; random forest settings, features, and validation folds held constant",submission_rf_grouped_age.csv
5,EXP-006,Compared ordinary stratified validation with family-group-aware validation,RandomForestClassifier,"Pclass, Sex, Age, SibSp, Parch, Fare, Embarked, Title, FamilySize, IsAlone, Deck, FarePerPerson",5-fold StratifiedGroupKFold using surname and family size,0.8270,nan,"No Kaggle submission was needed because the final tra

,experiment_id,description,model,features,validation_method,cv_accuracy,kaggle_score,notes,filename,cv_scores,cv_std,cv_kaggle_gap,changes,submission_file
0,EXP-001,XGBoost baseline with engineered features,XGBClassifier,"Pclass, Sex, Age, SibSp, Parch, Fare, Embarked, Title, FamilySize, IsAlone, Deck, FarePerPerson",5-fold StratifiedKFold,0.8372,0.76076,"Highest local CV so far, but large CV-to-Kaggle gap. May be fitting training-specific nonlinear patterns.",nan,"0.8659, 0.8483, 0.8034, 0.8258, 0.8427",0.0212,0.0765,Initial engineered-feature baseline,submission_xgb_baseline.csv
1,EXP-002,Logistic regression using identical engineered features,LogisticRegression,"Pclass, Sex, Age, SibSp, Parch, Fare, Embarked, Title, FamilySize, IsAlone, Deck, FarePerPerson",5-fold StratifiedKFold,0.8283,0.76555,Lower CV than XGBoost but better Kaggle score and smaller validation-to-leaderboard gap. Current leader.,nan,"0.8436, 0.8258, 0.7978, 0.8315, 0.8427",0.0167,0.0627,Replaced XGBClassifier with LogisticRegression; all other modeling steps held constant,submission_logistic.csv
2,EXP-003,Random forest using identical engineered features,RandomForestClassifier,"Pclass, Sex, Age, SibSp, Parch, Fare, Embarked, Title, FamilySize, IsAlone, Deck, FarePerPerson",5-fold StratifiedKFold,0.8316,0.78708,"Best Kaggle score so far. Random forest differed from logistic regression on only 15 of 418 passengers. In 12 of the 15 cases, logistic predicted survival while random forest predicted death; 10 of those passengers were male and 7 were first-class males. This suggests random forest captured interactions between sex, class, title, fare, and deck that additive logistic regression could not represent. Models agreed on 96.4% of test predictions.",nan,"0.8380, 0.8202, 0.8202, 0.8315, 0.8483",0.0108,0.0446,"Replaced logistic regression with a regularized random forest; features, preprocessing, and validation folds held constant",submission_random_forest.csv
3,EXP-004,Logistic regression with targeted categorical interaction features,LogisticRegression,"Pclass, Sex, Age, SibSp, Parch, Fare, Embarked, Title, FamilySize, IsAlone, Deck, FarePerPerson, Sex_Pclass, Sex_Title, Title_Pclass",5-fold StratifiedKFold,0.8339,0.76315,Interaction features raised mean CV accuracy from 0.8283 to 0.8339 but reduced Kaggle accuracy from 0.76555 to 0.76315. The CV standard deviation and CV-to-Kaggle gap also increased. The targeted interactions therefore fit the training folds better without transferring to the hidden test passengers. Random Forest remains the best model at 0.78708.,nan,"0.8268, 0.8539, 0.7978, 0.8427, 0.8483",0.0202,0.0707,"Added Sex_Pclass, Sex_Title, and Title_Pclass interaction features to baseline logistic regression; model settings, preprocessing approach, and validation folds held constant",submission_logistic_interactions.csv
4,EXP-005,Random forest with title-and-class-based age imputation,RandomForestClassifier,"Pclass, Sex, Age, SibSp, Parch, Fare, Embarked, Title, FamilySize, IsAlone, Deck, FarePerPerson",5-fold StratifiedKFold,0.8294,0.78229,"Grouped age imputation reduced mean CV accuracy from 0.8316 to 0.8294 and Kaggle accuracy from 0.78708 to 0.78229. It changed only 4 of 418 test predictions versus EXP-003. Although CV variability decreased slightly, the changed predictions did not improve hidden-test performance. Reject EXP-005; EXP-003 remains champion.",nan,"0.8380, 0.8146, 0.8258, 0.8315, 0.8371",0.0086,0.0471,"Replaced overall median age imputation with fold-safe grouped median imputation using Title and Pclass; random forest settings, features, and validation folds held constant",submission_rf_grouped_age.csv
5,EXP-006,Compared ordinary stratified validation with family-group-aware validation,RandomForestClassifier,"Pclass, Sex, Age, SibSp, Parch, Fare, Embarked, Title, FamilySize, IsAlone, Deck, FarePerPerson",5-fold StratifiedGroupKFold using surname and family size,0.8270,nan,"No Kaggle submission was needed because the final tra

NameError: name 'best_ensemble_exp010' is not defined